# 02. Limpieza, calidad y construcción de la base


In [21]:
# 1. Importar librerías

import json
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

from IPython.display import display

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
pd.set_option("display.max_colwidth", 180)

print("Librerías cargadas")

Librerías cargadas


In [22]:
# 2. Detectar rutas

RUTA_ACTUAL = Path.cwd().resolve()

if RUTA_ACTUAL.name.lower() == "notebooks":
    RUTA_PROYECTO = RUTA_ACTUAL.parent
elif (RUTA_ACTUAL / "datos").exists():
    RUTA_PROYECTO = RUTA_ACTUAL
else:
    RUTA_PROYECTO = RUTA_ACTUAL.parent

RUTA_INTERMEDIOS = RUTA_PROYECTO / "datos" / "intermedios"
RUTA_PROCESADOS = RUTA_PROYECTO / "datos" / "procesados"

for ruta in [RUTA_INTERMEDIOS, RUTA_PROCESADOS]:
    ruta.mkdir(parents=True, exist_ok=True)

print("Proyecto:", RUTA_PROYECTO)

Proyecto: D:\Users\LAURA PEREZ\Desktop\CIENCIA DE DATOS\PROYECTO SECOP_BARRANCABERMEJA


In [23]:
# 3. Cargar base del cuaderno 01

ARCHIVO_BASE = RUTA_INTERMEDIOS / "01_secop_ii_base_inicial.parquet"

df = pd.read_parquet(ARCHIVO_BASE)

print(f"Registros: {len(df):,}")
print(f"Columnas: {df.shape[1]}")

Registros: 37,574
Columnas: 48


In [24]:
# 4. Verificar columnas esenciales

COLUMNAS_ESENCIALES = [
    "nombre_entidad",
    "nit_entidad",
    "id_contrato",
    "estado_contrato",
    "descripcion_del_proceso",
    "tipo_de_contrato",
    "modalidad_de_contratacion",
    "justificacion_modalidad_de",
    "fecha_de_firma",
    "fecha_de_inicio_del_contrato",
    "fecha_de_fin_del_contrato",
    "tipodocproveedor",
    "documento_proveedor",
    "proveedor_adjudicado",
    "es_grupo",
    "valor_del_contrato",
    "es_alcaldia"
]

faltantes = [c for c in COLUMNAS_ESENCIALES if c not in df.columns]

if faltantes:
    raise ValueError(f"Faltan columnas esenciales: {faltantes}")

print("Columnas esenciales verificadas")

Columnas esenciales verificadas


In [25]:
# 5. Crear copia de trabajo

base = df.copy()
base["fila_origen"] = np.arange(len(base))

print(f"Base de trabajo: {len(base):,}")

Base de trabajo: 37,574


In [26]:
# 6. Funciones de normalización

def normalizar_texto(valor):
    if pd.isna(valor):
        return ""

    valor = str(valor).strip().upper()
    valor = unicodedata.normalize("NFKD", valor)
    valor = "".join(
        caracter
        for caracter in valor
        if not unicodedata.combining(caracter)
    )
    valor = re.sub(r"\s+", " ", valor)
    return valor


def normalizar_bool(valor):
    if pd.isna(valor):
        return False

    return normalizar_texto(valor) in {
        "TRUE", "1", "SI", "YES", "VERDADERO"
    }

print("Funciones listas")

Funciones listas


In [27]:
# 7. Normalizar textos clave

COLUMNAS_TEXTO = {
    "tipo_de_contrato": "tipo_contrato_norm",
    "modalidad_de_contratacion": "modalidad_norm",
    "justificacion_modalidad_de": "justificacion_norm",
    "descripcion_del_proceso": "descripcion_norm",
    "tipodocproveedor": "tipo_documento_norm",
    "proveedor_adjudicado": "proveedor_norm",
    "estado_contrato": "estado_norm"
}

for origen, destino in COLUMNAS_TEXTO.items():
    base[destino] = base[origen].map(normalizar_texto)

base["es_alcaldia"] = base["es_alcaldia"].map(normalizar_bool)
base["es_grupo_flag"] = base["es_grupo"].map(normalizar_bool)

print("Textos normalizados")

Textos normalizados


In [28]:
# 8. Normalizar documentos e ID

base["documento_normalizado"] = (
    base["documento_proveedor"]
    .fillna("")
    .astype(str)
    .str.upper()
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
    .str.replace(r"[^A-Z0-9]", "", regex=True)
    .replace("", pd.NA)
)

base["id_contrato_norm"] = (
    base["id_contrato"]
    .fillna("")
    .astype(str)
    .str.strip()
)

print("Documentos e ID normalizados")

# Normalizar NIT de la entidad
base["nit_entidad_normalizado"] = (
    base["nit_entidad"]
    .fillna("")
    .astype(str)
    .str.upper()
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
    .str.replace(r"[^A-Z0-9]", "", regex=True)
    .replace("", pd.NA)
)

Documentos e ID normalizados


In [29]:
# 9. Crear catálogo institucional para análisis por entidad

CATALOGO_ENTIDADES = {
    "890201900": {
        "entidad_analisis": "Alcaldia Distrital de Barrancabermeja",
        "grupo_entidad_analisis": "Administracion central"
    },
    "890270833": {
        "entidad_analisis": "Empresa de Desarrollo Urbano y Vivienda",
        "grupo_entidad_analisis": "Entidad local separada"
    },
    "829000477": {
        "entidad_analisis": "Instituto para el Fomento del Deporte y la Recreacion",
        "grupo_entidad_analisis": "Entidad local separada"
    },
    "890270948": {
        "entidad_analisis": "Inspeccion de Transito y Transporte",
        "grupo_entidad_analisis": "Entidad local separada"
    },
    "829001846": {
        "entidad_analisis": "Empresa Social del Estado Barrancabermeja",
        "grupo_entidad_analisis": "Entidad local separada"
    },
    "900136865": {
        "entidad_analisis": "Hospital Regional del Magdalena Medio",
        "grupo_entidad_analisis": "Entidad local separada"
    },
    "829001855": {
        "entidad_analisis": "Personeria de Barrancabermeja",
        "grupo_entidad_analisis": "Organo de control local"
    },
    "8290007456": {
        "entidad_analisis": "Contraloria de Barrancabermeja",
        "grupo_entidad_analisis": "Organo de control local"
    },
    "829001276": {
        "entidad_analisis": "Concejo de Barrancabermeja",
        "grupo_entidad_analisis": "Corporacion publica local"
    }
}

mapa_entidad = {
    nit: datos["entidad_analisis"]
    for nit, datos in CATALOGO_ENTIDADES.items()
}

mapa_grupo = {
    nit: datos["grupo_entidad_analisis"]
    for nit, datos in CATALOGO_ENTIDADES.items()
}

base["entidad_analisis"] = (
    base["nit_entidad_normalizado"]
    .map(mapa_entidad)
)

if "entidad_corta" in base.columns:
    base["entidad_analisis"] = (
        base["entidad_analisis"]
        .fillna(base["entidad_corta"])
    )

base["entidad_analisis"] = (
    base["entidad_analisis"]
    .fillna(base["nombre_entidad"])
)

base["grupo_entidad_analisis"] = (
    base["nit_entidad_normalizado"]
    .map(mapa_grupo)
    .fillna("Otra entidad")
)

base["es_alcaldia_nit"] = (
    base["nit_entidad_normalizado"].eq("890201900")
)

base["es_ese_barrancabermeja"] = (
    base["nit_entidad_normalizado"].eq("829001846")
    | base["nombre_entidad"].map(normalizar_texto).eq(
        "EMPRESA SOCIAL DEL ESTADO BARRANCABERMEJA"
    )
)

base["flag_inconsistencia_alcaldia"] = (
    base["es_alcaldia"].astype(bool)
    != base["es_alcaldia_nit"]
)

# Para análisis institucional se usa el NIT como criterio principal
base["es_alcaldia_analisis"] = base["es_alcaldia_nit"]

resumen_catalogo_entidades = (
    base.groupby(
        ["entidad_analisis", "grupo_entidad_analisis"],
        dropna=False
    )
    .agg(
        registros=("id_contrato", "size")
    )
    .reset_index()
    .sort_values("registros", ascending=False)
)

display(resumen_catalogo_entidades)

print(
    "Inconsistencias entre flag Alcaldia y NIT:",
    int(base["flag_inconsistencia_alcaldia"].sum())
)

,entidad_analisis,grupo_entidad_analisis,registros
0,Alcaldia Distrital de Barrancabermeja,Administracion central,27930
7,Instituto para el Fomento del Deporte y la Recreacion,Entidad local separada,2489
3,Empresa Social del Estado Barrancabermeja,Entidad local separada,2327
1,Concejo de Barrancabermeja,Corporacion publica local,1546
4,Empresa de Desarrollo Urbano y Vivienda,Entidad local separada,870
6,Inspeccion de Transito y Transporte,Entidad local separada,769
8,Personeria de Barrancabermeja,Organo de control local,659
5,Hospital Regional del Magdalena Medio,Entidad local separada,611
2,Contraloria de Barrancabermeja,Organo de control local,373


Inconsistencias entre flag Alcaldia y NIT: 0


In [30]:
# 10. Clasificar proveedor con reglas trazables

PATRON_NATURAL = (
    r"CEDULA DE CIUDADANIA|CEDULA DE EXTRANJERIA|PASAPORTE|"
    r"PERMISO POR PROTECCION TEMPORAL|REGISTRO CIVIL|TARJETA DE IDENTIDAD"
)

PATRON_GRUPO_NOMBRE = r"CONSORCIO|UNION TEMPORAL|PROMESA DE SOCIEDAD FUTURA"

PATRON_EMPRESA_NOMBRE = (
    r"S\.?A\.?S\.?(?:$|\s)|S\.?A\.?(?:$|\s)|LTDA|LIMITADA|"
    r"FUNDACION|CORPORACION|ASOCIACION|COOPERATIVA|HOGAR|"
    r"EMPRESA SOCIAL DEL ESTADO|E\.?S\.?E\.?"
)

DOCUMENTOS_NATURALES_VERIFICADOS = {
    "1192765565",
    "91446208",
    "52116279"
}

NOMBRES_NATURALES_VERIFICADOS = {
    "ZHARICK YUKSEL JARAMILLO CARRENO",
    "LUIS FERNANDO AVILA GUERRERO",
    "MELBA JOHANA VARON CARDENAS"
}

es_grupo_nombre = base["proveedor_norm"].str.contains(
    PATRON_GRUPO_NOMBRE, regex=True, na=False
)

es_natural_documento = base["tipo_documento_norm"].str.contains(
    PATRON_NATURAL, regex=True, na=False
)

es_juridica_documento = base["tipo_documento_norm"].eq("NIT")

es_natural_verificado = (
    base["documento_normalizado"].isin(DOCUMENTOS_NATURALES_VERIFICADOS)
    | base["proveedor_norm"].isin(NOMBRES_NATURALES_VERIFICADOS)
)

es_juridica_nombre_claro = base["proveedor_norm"].str.contains(
    PATRON_EMPRESA_NOMBRE, regex=True, na=False
)

base["tipo_proveedor"] = np.select(
    [
        base["es_grupo_flag"] | es_grupo_nombre,
        es_natural_documento,
        es_juridica_documento,
        es_natural_verificado,
        es_juridica_nombre_claro
    ],
    [
        "Grupo o consorcio",
        "Persona natural",
        "Persona juridica",
        "Persona natural",
        "Persona juridica"
    ],
    default="Por revisar"
)

base["tipo_proveedor_fuente"] = np.select(
    [
        base["es_grupo_flag"] | es_grupo_nombre,
        es_natural_documento,
        es_juridica_documento,
        es_natural_verificado,
        es_juridica_nombre_claro
    ],
    [
        "SECOP grupo o nombre",
        "Tipo documento SECOP",
        "Tipo documento SECOP",
        "Correccion manual verificada",
        "Regla juridica por nombre"
    ],
    default="Requiere revision"
)

base["confianza_tipo_proveedor"] = np.select(
    [
        base["tipo_proveedor_fuente"].isin([
            "Tipo documento SECOP",
            "Correccion manual verificada",
            "SECOP grupo o nombre"
        ]),
        base["tipo_proveedor_fuente"].eq("Regla juridica por nombre")
    ],
    ["Alta", "Media"],
    default="Baja"
)

print(base["tipo_proveedor"].value_counts(dropna=False))
print(base["confianza_tipo_proveedor"].value_counts(dropna=False))

tipo_proveedor
Persona natural      35036
Persona juridica      2354
Grupo o consorcio      184
Name: count, dtype: int64
confianza_tipo_proveedor
Alta     37547
Media       27
Name: count, dtype: int64


In [31]:
# 11. Crear llave del proveedor

base["proveedor_llave"] = np.where(
    base["documento_normalizado"].notna(),
    "DOC:" + base["documento_normalizado"].fillna(""),
    "NOMBRE:" + base["proveedor_norm"]
)

base.loc[
    base["documento_normalizado"].isna()
    & base["proveedor_norm"].eq(""),
    "proveedor_llave"
] = pd.NA

print(f"Proveedores identificados: {base['proveedor_llave'].nunique():,}")

Proveedores identificados: 11,328


In [32]:
# 12. Auditar identidad del proveedor

doc_nombres = (
    base.dropna(subset=["documento_normalizado"])
    .groupby("documento_normalizado")
    .agg(
        nombres_distintos=("proveedor_norm", "nunique"),
        registros=("id_contrato", "size")
    )
    .reset_index()
)

documentos_con_varios_nombres = doc_nombres.loc[
    doc_nombres["nombres_distintos"] > 1
].copy()

doc_tipos = (
    base.dropna(subset=["documento_normalizado"])
    .groupby("documento_normalizado")
    .agg(tipos_distintos=("tipo_proveedor", "nunique"))
    .reset_index()
)

documentos_con_varios_tipos = doc_tipos.loc[
    doc_tipos["tipos_distintos"] > 1
].copy()

nombre_docs = (
    base.loc[base["proveedor_norm"].ne("")]
    .groupby("proveedor_norm")
    .agg(documentos_distintos=("documento_normalizado", "nunique"))
    .reset_index()
)

nombres_con_varios_documentos = nombre_docs.loc[
    nombre_docs["documentos_distintos"] > 1
].copy()

documentos_normalizados = base["documento_normalizado"].fillna("")
longitud_doc = documentos_normalizados.str.len()
repetido_doc = documentos_normalizados.map(
    lambda valor: bool(valor) and len(set(valor)) == 1
)

base["flag_documento_sospechoso"] = (
    base["documento_normalizado"].notna()
    & (
        longitud_doc.lt(5)
        | longitud_doc.gt(15)
        | repetido_doc
    )
)

print(f"Documentos con varios nombres: {len(documentos_con_varios_nombres):,}")
print(f"Documentos con varios tipos: {len(documentos_con_varios_tipos):,}")
print(f"Nombres con varios documentos: {len(nombres_con_varios_documentos):,}")
print(f"Registros con documento sospechoso: {base['flag_documento_sospechoso'].sum():,}")

Documentos con varios nombres: 7
Documentos con varios tipos: 0
Nombres con varios documentos: 12
Registros con documento sospechoso: 1


In [33]:
# 13. Crear nombre canónico y flags de identidad

def nombre_mas_frecuente(serie):
    valores = (
        serie.dropna()
        .astype(str)
        .str.strip()
    )
    valores = valores[valores.ne("")]
    if valores.empty:
        return pd.NA
    return valores.value_counts().index[0]

nombre_canonico = (
    base.dropna(subset=["proveedor_llave"])
    .groupby("proveedor_llave")["proveedor_adjudicado"]
    .agg(nombre_mas_frecuente)
)

base["proveedor_nombre_canonico"] = (
    base["proveedor_llave"].map(nombre_canonico)
)

docs_varios_nombres_set = set(
    documentos_con_varios_nombres["documento_normalizado"]
)

base["flag_documento_varios_nombres"] = (
    base["documento_normalizado"].isin(docs_varios_nombres_set)
)

print(
    "Registros con documento asociado a varios nombres:",
    int(base["flag_documento_varios_nombres"].sum())
)

Registros con documento asociado a varios nombres: 65


In [34]:
# 14. Convertir fechas

COLUMNAS_FECHA = [
    "fecha_de_firma",
    "fecha_de_inicio_del_contrato",
    "fecha_de_fin_del_contrato"
]

for columna in COLUMNAS_FECHA:
    base[columna] = pd.to_datetime(base[columna], errors="coerce")

if "fecha_referencia" in base.columns:
    base["fecha_referencia"] = pd.to_datetime(
        base["fecha_referencia"], errors="coerce"
    )
else:
    base["fecha_referencia"] = pd.NaT

base["fecha_asignacion_periodo"] = (
    base["fecha_de_firma"]
    .combine_first(base["fecha_de_inicio_del_contrato"])
    .combine_first(base["fecha_referencia"])
)

base["fecha_asignacion_fuente"] = np.select(
    [
        base["fecha_de_firma"].notna(),
        base["fecha_de_inicio_del_contrato"].notna(),
        base["fecha_referencia"].notna()
    ],
    ["Fecha de firma", "Fecha de inicio", "Fecha de referencia"],
    default="Sin fecha"
)

print("Fechas convertidas")

Fechas convertidas


In [35]:
# 15. Convertir valores monetarios

COLUMNAS_VALOR = [
    "valor_del_contrato",
    "valor_facturado",
    "valor_pagado",
    "valor_pendiente_de_pago",
    "valor_pendiente_de_ejecucion"
]

for columna in COLUMNAS_VALOR:
    if columna in base.columns:
        base[columna + "_num"] = pd.to_numeric(
            base[columna], errors="coerce"
        )

base["valor_contrato_num"] = pd.to_numeric(
    base["valor_del_contrato"], errors="coerce"
)

base["flag_valor_negativo"] = base["valor_contrato_num"].lt(0).fillna(False)
base["flag_valor_cero"] = base["valor_contrato_num"].eq(0).fillna(False)
base["flag_valor_faltante"] = base["valor_contrato_num"].isna()

print("Valores convertidos")

Valores convertidos


In [36]:
# 16. Clasificar estado contractual

PATRON_ESTADO_EXCLUIR = (
    r"BORRADOR|DRAFT|CANCELAD|CANCELLED|ANULAD|ELIMINAD|"
    r"RECHAZAD|DESISTID|REVOCAD|ABORTAD"
)

PATRON_ESTADO_INCLUIR = (
    r"ACTIV|EN EJECUCION|EJECUCION|TERMINAD|CERRAD|LIQUIDAD|"
    r"SUSPENDID|MODIFICAD|CEDID|PRORROGAD|FINALIZAD"
)

estado_excluir = base["estado_norm"].str.contains(
    PATRON_ESTADO_EXCLUIR, regex=True, na=False
)

estado_incluir = base["estado_norm"].str.contains(
    PATRON_ESTADO_INCLUIR, regex=True, na=False
)

estado_aprobado = base["estado_norm"].eq("APROBADO")

aprobado_con_soporte = (
    estado_aprobado
    & base["fecha_de_firma"].notna()
    & base["proveedor_llave"].notna()
    & base["valor_contrato_num"].gt(0)
)

base["estado_analitico"] = np.select(
    [estado_excluir, estado_incluir, aprobado_con_soporte],
    ["Excluir", "Incluir", "Incluir"],
    default="Revisar"
)

base["criterio_estado"] = np.select(
    [estado_excluir, estado_incluir, aprobado_con_soporte, estado_aprobado],
    [
        "Estado excluido",
        "Estado contractual valido",
        "Aprobado con firma proveedor y valor positivo",
        "Aprobado sin soporte suficiente"
    ],
    default="Estado no clasificado"
)

base["contrato_valido_analisis"] = base["estado_analitico"].eq("Incluir")

print(base["estado_analitico"].value_counts(dropna=False))

estado_analitico
Incluir    37566
Revisar        8
Name: count, dtype: int64


In [37]:
# 17. Auditar estados

auditoria_estados = (
    base.groupby(
        ["estado_contrato", "estado_norm", "estado_analitico", "criterio_estado"],
        dropna=False
    )
    .agg(
        registros=("id_contrato", "size"),
        valor_total=("valor_contrato_num", "sum")
    )
    .reset_index()
    .sort_values("registros", ascending=False)
)

display(auditoria_estados)

,estado_contrato,estado_norm,estado_analitico,criterio_estado,registros,valor_total
2,Cerrado,CERRADO,Incluir,Estado contractual valido,19830,"507,736,833,167.99"
3,En ejecución,EN EJECUCION,Incluir,Estado contractual valido,10027,"300,123,263,684.67"
4,Modificado,MODIFICADO,Incluir,Estado contractual valido,4181,"858,619,350,109.38"
7,terminado,TERMINADO,Incluir,Estado contractual valido,3174,"261,978,551,368.96"
0,Aprobado,APROBADO,Incluir,Aprobado con firma proveedor y valor positivo,325,"15,806,325,313.67"
5,Suspendido,SUSPENDIDO,Incluir,Estado contractual valido,16,"44,985,792,058.74"
6,cedido,CEDIDO,Incluir,Estado contractual valido,13,"272,534,700.00"
1,Aprobado,APROBADO,Revisar,Aprobado sin soporte suficiente,8,0.00


In [38]:
# 18. Crear periodos metodológicos

CORTE_DATOS = pd.Timestamp("2026-09-06 23:59:59")

INICIO_ALFONSO_CONFIABLE = pd.Timestamp("2021-04-01")
FIN_ALFONSO = pd.Timestamp("2023-12-31 23:59:59")
INICIO_JONATHAN = pd.Timestamp("2024-01-01")

INICIO_32_ALFONSO = pd.Timestamp("2021-04-01")
FIN_32_ALFONSO = pd.Timestamp("2023-11-30 23:59:59")
INICIO_32_JONATHAN = pd.Timestamp("2024-01-01")
FIN_32_JONATHAN = pd.Timestamp("2026-08-31 23:59:59")

INICIO_24_ALFONSO = pd.Timestamp("2022-01-01")
FIN_24_ALFONSO = pd.Timestamp("2023-12-31 23:59:59")
INICIO_24_JONATHAN = pd.Timestamp("2024-01-01")
FIN_24_JONATHAN = pd.Timestamp("2025-12-31 23:59:59")

INICIO_ANIO3_ALFONSO = pd.Timestamp("2022-01-01")
FIN_ANIO3_ALFONSO = pd.Timestamp("2022-09-06 23:59:59")
INICIO_ANIO3_JONATHAN = pd.Timestamp("2026-01-01")
FIN_ANIO3_JONATHAN = pd.Timestamp("2026-09-06 23:59:59")

fecha = base["fecha_asignacion_periodo"]

base["alcalde"] = np.select(
    [
        base["es_alcaldia_analisis"] & fecha.between("2020-01-01", "2023-12-31 23:59:59"),
        base["es_alcaldia_analisis"] & fecha.between("2024-01-01", CORTE_DATOS)
    ],
    ["Alfonso Eljach", "Jonathan Vasquez"],
    default="No aplica"
)

base["cobertura_confiable_alcaldia"] = (
    base["es_alcaldia_analisis"]
    & (
        (
            base["alcalde"].eq("Alfonso Eljach")
            & fecha.between(INICIO_ALFONSO_CONFIABLE, FIN_ALFONSO)
        )
        |
        (
            base["alcalde"].eq("Jonathan Vasquez")
            & fecha.between(INICIO_JONATHAN, CORTE_DATOS)
        )
    )
)

base["periodo_comparable_32m"] = (
    (
        base["alcalde"].eq("Alfonso Eljach")
        & fecha.between(INICIO_32_ALFONSO, FIN_32_ALFONSO)
    )
    |
    (
        base["alcalde"].eq("Jonathan Vasquez")
        & fecha.between(INICIO_32_JONATHAN, FIN_32_JONATHAN)
    )
)

base["periodo_comparable_24m"] = (
    (
        base["alcalde"].eq("Alfonso Eljach")
        & fecha.between(INICIO_24_ALFONSO, FIN_24_ALFONSO)
    )
    |
    (
        base["alcalde"].eq("Jonathan Vasquez")
        & fecha.between(INICIO_24_JONATHAN, FIN_24_JONATHAN)
    )
)

base["periodo_anio3_mismo_corte"] = (
    (
        base["alcalde"].eq("Alfonso Eljach")
        & fecha.between(INICIO_ANIO3_ALFONSO, FIN_ANIO3_ALFONSO)
    )
    |
    (
        base["alcalde"].eq("Jonathan Vasquez")
        & fecha.between(INICIO_ANIO3_JONATHAN, FIN_ANIO3_JONATHAN)
    )
)

base["anio_periodo"] = fecha.dt.year.astype("Int64")
base["mes_periodo"] = fecha.dt.month.astype("Int64")

print("Periodos metodológicos creados")

Periodos metodológicos creados


In [39]:
# 18B. Año de gobierno, calendario electoral y ley de garantias

# Variables derivadas descritas en docs/proyecto-objetivos.md (seccion 8):
# anio_gobierno, periodo_electoral y periodo_preelectoral.
# Se agregan ademas mes_gobierno (posicion exacta dentro del mandato),
# ventana_ley_garantias (restriccion legal a la contratacion directa antes de
# cada eleccion territorial) y dias_a_prox_eleccion.

# ---------------------------------------------------------------
# A. Posicion del contrato dentro del mandato
# ---------------------------------------------------------------
# Los mandatos de alcalde en Colombia inician el 1 de enero.
INICIO_MANDATO = {
    "Alfonso Eljach": pd.Timestamp("2020-01-01"),
    "Jonathan Vasquez": pd.Timestamp("2024-01-01")
}

anio_inicio_mandato = base["alcalde"].map(
    {nombre: fecha.year for nombre, fecha in INICIO_MANDATO.items()}
)
mes_inicio_mandato = base["alcalde"].map(
    {nombre: fecha.month for nombre, fecha in INICIO_MANDATO.items()}
)

base["mes_gobierno"] = (
    (base["fecha_asignacion_periodo"].dt.year - anio_inicio_mandato) * 12
    + (base["fecha_asignacion_periodo"].dt.month - mes_inicio_mandato)
    + 1
).astype("Int64")

base["anio_gobierno"] = (
    (base["mes_gobierno"] - 1) // 12 + 1
).astype("Int64")

# Las entidades distintas de la Alcaldia quedan sin mes de gobierno (alcalde = No aplica)

# ---------------------------------------------------------------
# B. Calendario electoral y ventana de ley de garantias
# ---------------------------------------------------------------
ELECCIONES_TERRITORIALES = [
    pd.Timestamp("2019-10-27"),
    pd.Timestamp("2023-10-29"),
    pd.Timestamp("2027-10-29")
]

MESES_RESTRICCION_GARANTIAS = 4  # meses previos a la eleccion con restriccion legal


def dias_a_proxima_eleccion(fecha_valor):
    if pd.isna(fecha_valor):
        return np.nan
    futuras = [
        (eleccion - fecha_valor).days
        for eleccion in ELECCIONES_TERRITORIALES
        if eleccion >= fecha_valor
    ]
    return min(futuras) if futuras else np.nan


def en_ventana_ley_garantias(fecha_valor):
    if pd.isna(fecha_valor):
        return False
    for eleccion in ELECCIONES_TERRITORIALES:
        inicio_ventana = eleccion - pd.DateOffset(months=MESES_RESTRICCION_GARANTIAS)
        if inicio_ventana <= fecha_valor <= eleccion:
            return True
    return False


base["dias_a_prox_eleccion"] = base["fecha_asignacion_periodo"].map(
    dias_a_proxima_eleccion
)
base["ventana_ley_garantias"] = base["fecha_asignacion_periodo"].map(
    en_ventana_ley_garantias
)

ANIOS_ELECTORALES = {2019, 2023, 2027}
ANIOS_PREELECTORALES = {2018, 2022, 2026}

base["periodo_electoral"] = base["anio_periodo"].isin(ANIOS_ELECTORALES)
base["periodo_preelectoral"] = base["anio_periodo"].isin(ANIOS_PREELECTORALES)

base["tipo_anio_electoral"] = np.select(
    [base["periodo_electoral"], base["periodo_preelectoral"]],
    ["Electoral", "Preelectoral"],
    default="Ordinario"
)

print("Año de gobierno, calendario electoral y ley de garantias creados")
print()
print("Contratos por año de gobierno:")
print(
    base.loc[base["alcalde"].ne("No aplica")]
    .groupby(["alcalde", "anio_gobierno"])
    .size()
)
print()
print(base["tipo_anio_electoral"].value_counts(dropna=False))
print(
    "Contratos en ventana de ley de garantias:",
    int(base["ventana_ley_garantias"].sum())
)

Año de gobierno, calendario electoral y ley de garantias creados

Contratos por año de gobierno:
alcalde           anio_gobierno
Alfonso Eljach    1                   7
                  2                3035
                  3                5855
                  4                4790
Jonathan Vasquez  1                4853
                  2                4991
                  3                4399
dtype: int64

tipo_anio_electoral
Ordinario       17562
Preelectoral    13769
Electoral        6243
Name: count, dtype: int64
Contratos en ventana de ley de garantias: 184


In [40]:
# 19. Calcular duración contractual

base["flag_fecha_invertida"] = (
    base["fecha_de_inicio_del_contrato"].notna()
    & base["fecha_de_fin_del_contrato"].notna()
    & (
        base["fecha_de_inicio_del_contrato"]
        > base["fecha_de_fin_del_contrato"]
    )
)

base["duracion_dias"] = (
    base["fecha_de_fin_del_contrato"]
    - base["fecha_de_inicio_del_contrato"]
).dt.days

base.loc[base["flag_fecha_invertida"], "duracion_dias"] = np.nan
base.loc[base["duracion_dias"] < 0, "duracion_dias"] = np.nan

base["duracion_meses_exacta"] = base["duracion_dias"] / 30.44
base["duracion_meses"] = base["duracion_meses_exacta"]

base["valor_mensual_equivalente"] = np.where(
    base["duracion_meses_exacta"].gt(0)
    & base["valor_contrato_num"].gt(0),
    base["valor_contrato_num"] / base["duracion_meses_exacta"],
    np.nan
)

# Duracion cero: imposible en un contrato con valor. Indica que la fecha de fin
# no fue diligenciada y quedo igual a la de inicio. No se imputa: solo se marca.
base["flag_duracion_cero"] = (
    base["duracion_dias"].notna()
    & base["duracion_dias"].le(0)
)

# ---------------------------------------------------------------
# Valor mensual ajustado por la ejecucion real del contrato
# ---------------------------------------------------------------
# SECOP registra la fecha de fin REAL. Cuando un contrato se termina
# anticipadamente, la duracion queda corta pero el valor del contrato sigue
# siendo el total pactado. Dividir el total entre la duracion corta infla el
# valor mensual (se han visto sobreestimaciones de hasta 3 veces).
#
# Regla: si el contrato YA TERMINO y quedo valor sin ejecutar, la referencia
# correcta para el valor mensual es el valor efectivamente ejecutado, que
# corresponde al mismo periodo que la duracion registrada.

UMBRAL_EJECUCION = 0.95

base["valor_ejecutado_num"] = base["valor_pagado_num"].where(
    base["valor_pagado_num"].gt(0),
    base["valor_facturado_num"]
)

base["pct_ejecucion"] = np.where(
    base["valor_contrato_num"].gt(0),
    base["valor_ejecutado_num"] / base["valor_contrato_num"],
    np.nan
)

base["contrato_ya_termino"] = base["fecha_de_fin_del_contrato"].le(CORTE_DATOS)

# El ajuste SOLO se aplica cuando el estado indica cierre definitivo.
#
# Motivo (verificado empiricamente sobre estos datos): el reporte de pagos en
# SECOP llega con rezago. Entre los contratos que NO estan cerrados, la tasa de
# ejecucion parcial cae de 50% (terminados hace menos de 6 meses) a 15% (hace
# 1 a 2 anos): eso es rezago, no terminacion anticipada. En cambio, entre los
# contratos CERRADOS la tasa es estable (5,5% a 9,2%, sin tendencia) y casi
# igual para ambos alcaldes (7,3% y 6,5%): esa si es terminacion anticipada real.
#
# Sin esta condicion el ajuste castigaria a la administracion con contratos mas
# recientes por el solo hecho de serlo, introduciendo un sesgo grave.

ESTADOS_CIERRE_DEFINITIVO = {"CERRADO", "TERMINADO", "LIQUIDADO"}

base["estado_cierre_definitivo"] = base["estado_norm"].isin(ESTADOS_CIERRE_DEFINITIVO)

base["flag_ejecucion_parcial"] = (
    base["contrato_ya_termino"]
    & base["estado_cierre_definitivo"]
    & base["pct_ejecucion"].lt(UMBRAL_EJECUCION)
    & base["valor_ejecutado_num"].gt(0)
)

# Cerrado sin ningun valor ejecutado reportado: no hay base para estimar
base["flag_valor_mensual_no_confiable"] = (
    base["contrato_ya_termino"]
    & base["estado_cierre_definitivo"]
    & base["valor_ejecutado_num"].fillna(0).le(0)
    & base["valor_contrato_num"].gt(0)
)

valor_base_mensual = np.where(
    base["flag_ejecucion_parcial"],
    base["valor_ejecutado_num"],
    base["valor_contrato_num"]
)

base["valor_mensual_ajustado"] = np.where(
    base["duracion_meses_exacta"].gt(0) & (valor_base_mensual > 0),
    valor_base_mensual / base["duracion_meses_exacta"],
    np.nan
)

base["fuente_valor_mensual"] = np.select(
    [
        base["flag_valor_mensual_no_confiable"],
        base["flag_ejecucion_parcial"]
    ],
    [
        "Sin base confiable",
        "Valor ejecutado (terminacion anticipada)"
    ],
    default="Valor del contrato"
)

print("Duración y valor mensual calculados")
print(
    "Contratos con duracion cero (fecha de fin no diligenciada):",
    int(base["flag_duracion_cero"].sum())
)
print(
    "Contratos con ejecucion parcial (valor mensual ajustado):",
    int(base["flag_ejecucion_parcial"].sum())
)
print(
    "Contratos sin base confiable para valor mensual:",
    int(base["flag_valor_mensual_no_confiable"].sum())
)
print()
print(base["fuente_valor_mensual"].value_counts().to_string())

Duración y valor mensual calculados
Contratos con duracion cero (fecha de fin no diligenciada): 107
Contratos con ejecucion parcial (valor mensual ajustado): 1458
Contratos sin base confiable para valor mensual: 402

fuente_valor_mensual
Valor del contrato                          35714
Valor ejecutado (terminacion anticipada)     1458
Sin base confiable                            402


In [41]:
# 20. Clasificar duración en meses aproximados

meses_aprox = (base["duracion_dias"] / 30.44).round().astype("Int64")
base["duracion_meses_aprox"] = meses_aprox


def etiqueta_duracion_aprox(valor):
    if pd.isna(valor):
        return "Sin duración calculable"

    valor = int(valor)
    if valor <= 0:
        return "Menos de 1 mes"
    if valor == 1:
        return "1 mes aprox."
    if 2 <= valor <= 12:
        return f"{valor} meses aprox."
    if 13 <= valor <= 18:
        return "13 a 18 meses aprox."
    if 19 <= valor <= 24:
        return "19 a 24 meses aprox."
    return "Mas de 24 meses"


orden_duracion_clara = [
    "Menos de 1 mes",
    "1 mes aprox.",
    "2 meses aprox.",
    "3 meses aprox.",
    "4 meses aprox.",
    "5 meses aprox.",
    "6 meses aprox.",
    "7 meses aprox.",
    "8 meses aprox.",
    "9 meses aprox.",
    "10 meses aprox.",
    "11 meses aprox.",
    "12 meses aprox.",
    "13 a 18 meses aprox.",
    "19 a 24 meses aprox.",
    "Mas de 24 meses",
    "Sin duración calculable",
]

base["duracion_categoria_clara"] = pd.Categorical(
    base["duracion_meses_aprox"].map(etiqueta_duracion_aprox),
    categories=orden_duracion_clara,
    ordered=True,
)

print(
    base["duracion_categoria_clara"]
    .value_counts(dropna=False)
    .sort_index()
)

duracion_categoria_clara
Menos de 1 mes              791
1 mes aprox.               3699
2 meses aprox.             5504
3 meses aprox.             9348
4 meses aprox.             8662
5 meses aprox.             3787
6 meses aprox.             3662
7 meses aprox.              558
8 meses aprox.              344
9 meses aprox.              313
10 meses aprox.             203
11 meses aprox.             167
12 meses aprox.              81
13 a 18 meses aprox.         42
19 a 24 meses aprox.         23
Mas de 24 meses              57
Sin duración calculable     333
Name: count, dtype: int64


In [42]:
# 21. Auditar días adicionados

if "dias_adicionados" in base.columns:
    base["dias_adicionados_num"] = pd.to_numeric(
        base["dias_adicionados"], errors="coerce"
    )
else:
    base["dias_adicionados_num"] = np.nan

base["flag_tiene_dias_adicionados"] = (
    base["dias_adicionados_num"].fillna(0).gt(0)
)

print(
    "Contratos con días adicionados:",
    int(base["flag_tiene_dias_adicionados"].sum())
)

Contratos con días adicionados: 3632


In [43]:
# 22. Crear llave de contrato y deduplicar

base["contrato_llave"] = np.where(
    base["id_contrato_norm"].ne(""),
    "ID:" + base["id_contrato_norm"],
    "FILA:" + base["fila_origen"].astype(str)
)

COLUMNAS_COMPLETITUD = [
    "id_contrato",
    "documento_normalizado",
    "proveedor_adjudicado",
    "tipo_de_contrato",
    "modalidad_de_contratacion",
    "fecha_de_firma",
    "fecha_de_inicio_del_contrato",
    "fecha_de_fin_del_contrato",
    "valor_contrato_num"
]

base["puntaje_completitud"] = base[COLUMNAS_COMPLETITUD].notna().sum(axis=1)

ids_repetidos = base.loc[
    base["id_contrato_norm"].ne("")
    & base["id_contrato_norm"].duplicated(keep=False)
].copy()

base_contratos = (
    base.sort_values("puntaje_completitud", ascending=False)
    .drop_duplicates(subset="contrato_llave", keep="first")
    .reset_index(drop=True)
)

print(f"Filas con ID repetido antes de resolver: {len(ids_repetidos):,}")
print(f"Contratos únicos: {len(base_contratos):,}")

Filas con ID repetido antes de resolver: 0
Contratos únicos: 37,574


In [44]:
# 23. Crear flags de aptitud analítica

base_contratos["apto_conteo_contratos"] = (
    base_contratos["contrato_valido_analisis"]
    & base_contratos["contrato_llave"].notna()
)

base_contratos["apto_conteo_personas"] = (
    base_contratos["apto_conteo_contratos"]
    & base_contratos["proveedor_llave"].notna()
)

base_contratos["apto_analisis_monetario"] = (
    base_contratos["apto_conteo_contratos"]
    & base_contratos["valor_contrato_num"].gt(0)
)

base_contratos["apto_analisis_duracion"] = (
    base_contratos["apto_conteo_contratos"]
    & base_contratos["duracion_dias"].gt(0)
    & ~base_contratos["flag_fecha_invertida"]
)

base_contratos["apto_valor_mensual"] = (
    base_contratos["apto_analisis_monetario"]
    & base_contratos["apto_analisis_duracion"]
    & base_contratos["valor_mensual_equivalente"].notna()
)

print("Flags analíticos creados")

Flags analíticos creados


In [45]:
# 24. Identificar CPS estricto y ampliado

es_prestacion_servicios = (
    base_contratos["tipo_contrato_norm"]
    .str.contains("PRESTACION DE SERVICIOS", regex=False, na=False)
)

es_persona_natural = base_contratos["tipo_proveedor"].eq("Persona natural")

PATRON_EVIDENCIA_CPS = (
    r"SERVICIOS? PROFESIONALES?|"
    r"APOYO A LA GESTION|"
    r"SERVICIOS? DE APOYO"
)

evidencia_justificacion = base_contratos["justificacion_norm"].str.contains(
    PATRON_EVIDENCIA_CPS, regex=True, na=False
)

evidencia_descripcion = base_contratos["descripcion_norm"].str.contains(
    PATRON_EVIDENCIA_CPS, regex=True, na=False
)

base_contratos["es_cps_ampliado_semantico"] = (
    es_prestacion_servicios & es_persona_natural
)

base_contratos["es_cps_estricto_semantico"] = (
    base_contratos["es_cps_ampliado_semantico"]
    & (evidencia_justificacion | evidencia_descripcion)
)

base_contratos["es_cps_estricto"] = (
    base_contratos["es_cps_estricto_semantico"]
    & base_contratos["contrato_valido_analisis"]
)

base_contratos["es_cps_ampliado"] = (
    base_contratos["es_cps_ampliado_semantico"]
    & base_contratos["contrato_valido_analisis"]
)

base_contratos["clasificacion_cps"] = np.select(
    [
        base_contratos["es_cps_estricto_semantico"],
        base_contratos["es_cps_ampliado_semantico"]
    ],
    ["CPS estricto", "CPS probable"],
    default="No CPS"
)

print(base_contratos["clasificacion_cps"].value_counts())

clasificacion_cps
CPS estricto    32778
No CPS           4709
CPS probable       87
Name: count, dtype: int64


In [46]:
# 25. Separar profesional y apoyo a la gestión

PATRON_PROFESIONAL = r"SERVICIOS? PROFESIONALES?"
PATRON_APOYO = r"APOYO A LA GESTION|SERVICIOS? DE APOYO"

desc_profesional = base_contratos["descripcion_norm"].str.contains(
    PATRON_PROFESIONAL, regex=True, na=False
)
desc_apoyo = base_contratos["descripcion_norm"].str.contains(
    PATRON_APOYO, regex=True, na=False
)
just_profesional = base_contratos["justificacion_norm"].str.contains(
    PATRON_PROFESIONAL, regex=True, na=False
)
just_apoyo = base_contratos["justificacion_norm"].str.contains(
    PATRON_APOYO, regex=True, na=False
)

cond_prof_desc = desc_profesional & ~desc_apoyo
cond_apoyo_desc = desc_apoyo & ~desc_profesional
cond_mixto_desc = desc_profesional & desc_apoyo

sin_evidencia_desc = ~desc_profesional & ~desc_apoyo
cond_prof_just = sin_evidencia_desc & just_profesional & ~just_apoyo
cond_apoyo_just = sin_evidencia_desc & just_apoyo & ~just_profesional
cond_just_general = sin_evidencia_desc & just_profesional & just_apoyo

base_contratos["tipo_cps"] = np.select(
    [
        cond_prof_desc,
        cond_apoyo_desc,
        cond_mixto_desc,
        cond_prof_just,
        cond_apoyo_just,
        cond_just_general
    ],
    [
        "Profesional claro",
        "Apoyo a la gestion claro",
        "Mixto o ambiguo en descripcion",
        "Profesional claro",
        "Apoyo a la gestion claro",
        "Justificacion general sin subtipo"
    ],
    default="Sin evidencia suficiente"
)

base_contratos["tipo_cps_fuente"] = np.select(
    [
        cond_prof_desc,
        cond_apoyo_desc,
        cond_mixto_desc,
        cond_prof_just,
        cond_apoyo_just,
        cond_just_general
    ],
    [
        "Descripcion",
        "Descripcion",
        "Descripcion ambigua",
        "Justificacion unica",
        "Justificacion unica",
        "Justificacion general"
    ],
    default="Sin evidencia suficiente"
)

base_contratos.loc[
    ~base_contratos["es_cps_ampliado_semantico"],
    ["tipo_cps", "tipo_cps_fuente"]
] = ["No aplica", "No aplica"]

base_contratos["tipo_cps_claro"] = base_contratos["tipo_cps"].isin(
    ["Profesional claro", "Apoyo a la gestion claro"]
)

print(
    base_contratos.loc[
        base_contratos["es_cps_ampliado_semantico"],
        "tipo_cps"
    ].value_counts()
)

tipo_cps
Profesional claro                    15515
Apoyo a la gestion claro             14816
Justificacion general sin subtipo     2032
Mixto o ambiguo en descripcion         415
Sin evidencia suficiente                87
Name: count, dtype: int64


In [47]:
# 26. Clasificar familias contractuales con jerarquía metodológica

tipo = base_contratos["tipo_contrato_norm"]
modalidad = base_contratos["modalidad_norm"]
descripcion = base_contratos["descripcion_norm"]
justificacion = base_contratos["justificacion_norm"]

es_juridica_o_grupo = base_contratos["tipo_proveedor"].isin(
    ["Persona juridica", "Grupo o consorcio"]
)

es_natural = base_contratos["tipo_proveedor"].eq(
    "Persona natural"
)

# Evidencia explícita de prestación de servicios en el objeto
descripcion_prestacion_explicita = (
    descripcion.str.contains(
        r"\bPRESTACION DE SERVICIOS\b",
        regex=True,
        na=False
    )
)

# Servicios personales cuyo tipo SECOP fue diligenciado con otra categoría
base_contratos["es_servicio_natural_por_objeto"] = (
    es_natural
    & descripcion_prestacion_explicita
    & ~base_contratos["es_cps_ampliado_semantico"]
)

base_contratos["es_decreto_092"] = (
    tipo.str.contains(r"DECRETO 092", regex=True, na=False)
    | modalidad.str.contains(r"DECRETO 092", regex=True, na=False)
    | justificacion.str.contains(r"DECRETO 092", regex=True, na=False)
)

base_contratos["es_convenio_asociacion"] = (
    tipo.str.contains(
        r"CONVENIO DE ASOCIACION",
        regex=True,
        na=False
    )
    | justificacion.str.contains(
        r"CONVENIO DE ASOCIACION",
        regex=True,
        na=False
    )
)

base_contratos["menciona_esal"] = (
    tipo.str.contains(
        r"ENTIDAD SIN ANIMO DE LUCRO|\bESAL\b",
        regex=True,
        na=False
    )
    | justificacion.str.contains(
        r"ENTIDAD SIN ANIMO DE LUCRO|\bESAL\b",
        regex=True,
        na=False
    )
)

marcador_esal = (
    base_contratos["es_decreto_092"]
    | base_contratos["es_convenio_asociacion"]
    | base_contratos["menciona_esal"]
)

# ESAL solo se usa como familia cuando el proveedor es jurídico o grupo
es_esal_confirmada = (
    marcador_esal
    & es_juridica_o_grupo
)

# Persona natural bajo régimen especial que no tiene objeto explícito de servicios
es_regimen_especial_natural_revisar = (
    es_natural
    & modalidad.str.contains(
        "REGIMEN ESPECIAL",
        regex=False,
        na=False
    )
    & ~base_contratos["es_cps_ampliado_semantico"]
    & ~base_contratos["es_servicio_natural_por_objeto"]
)

es_interadministrativo = (
    tipo.str.contains(
        r"INTERADMINISTRATIVO|CONVENIO",
        regex=True,
        na=False
    )
    | modalidad.str.contains(
        r"INTERADMINISTRATIVO",
        regex=True,
        na=False
    )
    | justificacion.str.contains(
        r"INTERADMINISTRATIVO",
        regex=True,
        na=False
    )
)

es_app = tipo.str.contains(
    r"ASOCIACION PUBLICO PRIVADA",
    regex=True,
    na=False
)

es_enajenacion = tipo.str.contains(
    r"ENAJENACION|VENTA DE BIENES",
    regex=True,
    na=False
)

es_obra = tipo.str.contains(
    r"\bOBRA\b",
    regex=True,
    na=False
)

es_consultoria = tipo.str.contains(
    r"CONSULTORIA|INTERVENTORIA",
    regex=True,
    na=False
)

es_bienes = tipo.str.contains(
    r"SUMINISTRO|COMPRAVENTA|ADQUISICION DE BIENES",
    regex=True,
    na=False
)

es_arrendamiento = tipo.str.contains(
    r"ARRENDAMIENTO",
    regex=True,
    na=False
)

es_seguro = tipo.str.contains(
    r"\bSEGUROS?\b",
    regex=True,
    na=False
)

es_comodato = tipo.str.contains(
    r"COMODATO",
    regex=True,
    na=False
)

es_fiducia = tipo.str.contains(
    r"FIDUCI|ENCARGO FIDUCIARIO",
    regex=True,
    na=False
)

es_concesion = tipo.str.contains(
    r"CONCESION",
    regex=True,
    na=False
)

es_servicio_empresa = (
    tipo.str.contains(
        "PRESTACION DE SERVICIOS",
        regex=False,
        na=False
    )
    & es_juridica_o_grupo
)

es_regimen_especial = modalidad.str.contains(
    "REGIMEN ESPECIAL",
    regex=False,
    na=False
)

es_regimen_especial_juridico = (
    es_regimen_especial
    & es_juridica_o_grupo
)

base_contratos["familia_contrato"] = np.select(
    [
        base_contratos["es_cps_estricto_semantico"],
        (
            base_contratos["es_cps_ampliado_semantico"]
            & ~base_contratos["es_cps_estricto_semantico"]
        ),
        base_contratos["es_servicio_natural_por_objeto"],
        es_esal_confirmada,
        es_regimen_especial_natural_revisar,
        es_interadministrativo,
        es_app,
        es_enajenacion,
        es_obra,
        es_consultoria,
        es_bienes,
        es_arrendamiento,
        es_seguro,
        es_comodato,
        es_fiducia,
        es_concesion,
        es_servicio_empresa,
        es_regimen_especial_juridico
    ],
    [
        "CPS persona natural estricto",
        "Prestacion servicios persona natural por revisar",
        "Prestacion servicios persona natural por objeto explicito",
        "Entidad sin animo de lucro (ESAL) / Decreto 092",
        "Regimen especial persona natural por revisar",
        "Convenios e interadministrativos",
        "Asociacion publico privada",
        "Enajenacion de bienes",
        "Obra",
        "Consultoria e interventoria",
        "Bienes y suministros",
        "Arrendamiento",
        "Seguros",
        "Comodato",
        "Fiducia",
        "Concesion",
        "Servicios empresa o grupo",
        "Regimen especial persona juridica o grupo"
    ],
    default="Otros por revisar"
)

print(
    base_contratos["familia_contrato"]
    .value_counts()
)

familia_contrato
CPS persona natural estricto                                 32778
Prestacion servicios persona natural por objeto explicito     1827
Entidad sin animo de lucro (ESAL) / Decreto 092                915
Servicios empresa o grupo                                      732
Bienes y suministros                                           387
Arrendamiento                                                  173
Regimen especial persona natural por revisar                   171
Regimen especial persona juridica o grupo                      117
Obra                                                            97
Consultoria e interventoria                                     94
Prestacion servicios persona natural por revisar                87
Convenios e interadministrativos                                84
Comodato                                                        69
Seguros                                                         38
Otros por revisar                            

In [48]:
# 27. Auditar integridad entre CPS y familia contractual

integridad_cps_familia = (
    base_contratos.loc[
        base_contratos["es_cps_estricto_semantico"]
        & ~base_contratos["familia_contrato"].eq(
            "CPS persona natural estricto"
        ),
        [
            "id_contrato",
            "tipo_de_contrato",
            "modalidad_de_contratacion",
            "descripcion_del_proceso",
            "tipo_proveedor",
            "familia_contrato"
        ]
    ]
    .copy()
)

cruce_cps_familia = pd.crosstab(
    base_contratos["clasificacion_cps"],
    base_contratos["familia_contrato"]
)

print(
    "CPS estrictos fuera de su familia:",
    len(integridad_cps_familia)
)

display(cruce_cps_familia)

CPS estrictos fuera de su familia: 0


familia_contrato,Arrendamiento,Asociacion publico privada,Bienes y suministros,CPS persona natural estricto,Comodato,Consultoria e interventoria,Convenios e interadministrativos,Entidad sin animo de lucro (ESAL) / Decreto 092,Obra,Otros por revisar,Prestacion servicios persona natural por objeto explicito,Prestacion servicios persona natural por revisar,Regimen especial persona juridica o grupo,Regimen especial persona natural por revisar,Seguros,Servicios empresa o grupo
clasificacion_cps,,,,,,,,,,,,,,,,
CPS estricto,0,0,0,32778,0,0,0,0,0,0,0,0,0,0,0,0
CPS probable,0,0,0,0,0,0,0,0,0,0,0,87,0,0,0,0
No CPS,173,2,387,0,69,94,84,915,97,3,1827,0,117,171,38,732


In [49]:
# 28. Identificar mínima cuantía

base_contratos["es_minima_cuantia"] = (
    base_contratos["modalidad_norm"]
    .str.contains("MINIMA CUANTIA", na=False)
)

print(base_contratos["es_minima_cuantia"].value_counts())

es_minima_cuantia
False    37148
True       426
Name: count, dtype: int64


In [50]:
# 29. Crear base analítica válida

base_analitica = base_contratos.loc[
    base_contratos["contrato_valido_analisis"]
].copy()

registros_excluidos_estado = base_contratos.loc[
    base_contratos["estado_analitico"].eq("Excluir")
].copy()

registros_estado_revisar = base_contratos.loc[
    base_contratos["estado_analitico"].eq("Revisar")
].copy()

print(f"Base maestra: {len(base_contratos):,}")
print(f"Base analítica válida: {len(base_analitica):,}")
print(f"Excluidos por estado: {len(registros_excluidos_estado):,}")
print(f"Estados por revisar: {len(registros_estado_revisar):,}")

Base maestra: 37,574
Base analítica válida: 37,566
Excluidos por estado: 0
Estados por revisar: 8


In [51]:
# 30. Crear universos principales

cps = base_analitica.loc[
    base_analitica["es_cps_estricto"]
].copy()

cps_ampliado = base_analitica.loc[
    base_analitica["es_cps_ampliado"]
].copy()

contratistas_empresariales = base_analitica.loc[
    base_analitica["tipo_proveedor"].isin([
        "Persona juridica", "Grupo o consorcio"
    ])
].copy()

print(f"CPS estricto: {len(cps):,}")
print(f"Personas únicas CPS: {cps['proveedor_llave'].nunique():,}")
print(f"CPS ampliado: {len(cps_ampliado):,}")
print(f"Contratos empresariales: {len(contratistas_empresariales):,}")

CPS estricto: 32,778
Personas únicas CPS: 9,926
CPS ampliado: 32,865
Contratos empresariales: 2,535


In [52]:
# 31. Crear universo de servicios de persona natural por entidad

servicios_persona_natural_entidades = (
    base_analitica.loc[
        base_analitica["tipo_proveedor"].eq(
            "Persona natural"
        )
        & (
            base_analitica["es_cps_ampliado_semantico"]
            | base_analitica["es_servicio_natural_por_objeto"]
        )
    ]
    .copy()
)

servicios_persona_natural_entidades[
    "criterio_servicio_persona_natural"
] = np.select(
    [
        servicios_persona_natural_entidades[
            "es_cps_ampliado_semantico"
        ],
        servicios_persona_natural_entidades[
            "es_servicio_natural_por_objeto"
        ]
    ],
    [
        "Tipo SECOP Prestacion de servicios",
        "Objeto contractual explicito"
    ],
    default="Por revisar"
)

resumen_servicios_entidades = (
    servicios_persona_natural_entidades
    .groupby(
        "entidad_analisis",
        dropna=False
    )
    .agg(
        contratos=("contrato_llave", "nunique"),
        personas=("proveedor_llave", "nunique"),
        valor_total=("valor_contrato_num", "sum"),
        contratos_tipo_secop=(
            "es_cps_ampliado_semantico",
            "sum"
        ),
        contratos_por_objeto=(
            "es_servicio_natural_por_objeto",
            "sum"
        )
    )
    .reset_index()
    .sort_values(
        "contratos",
        ascending=False
    )
)

display(resumen_servicios_entidades)

servicios_ese = (
    servicios_persona_natural_entidades.loc[
        servicios_persona_natural_entidades[
            "entidad_analisis"
        ].eq(
            "Empresa Social del Estado Barrancabermeja"
        )
    ]
    .copy()
)

resumen_servicios_ese_anual = (
    servicios_ese.groupby(
        "anio_periodo",
        dropna=False
    )
    .agg(
        contratos=("contrato_llave", "nunique"),
        personas=("proveedor_llave", "nunique"),
        valor_total=("valor_contrato_num", "sum")
    )
    .reset_index()
    .sort_values("anio_periodo")
)

print(
    "Servicios de persona natural en ESE Barrancabermeja:",
    len(servicios_ese)
)

display(resumen_servicios_ese_anual)

,entidad_analisis,contratos,personas,valor_total,contratos_tipo_secop,contratos_por_objeto
0,Alcaldia Distrital de Barrancabermeja,26338,8728,"309,393,778,984.33",26337,1
7,Instituto para el Fomento del Deporte y la Recreacion,2262,771,"24,674,579,658.00",2261,1
3,Empresa Social del Estado Barrancabermeja,1824,1024,"23,072,202,995.29",0,1824
1,Concejo de Barrancabermeja,1509,835,"19,722,177,517.97",1509,0
4,Empresa de Desarrollo Urbano y Vivienda,785,313,"12,735,914,322.00",784,1
6,Inspeccion de Transito y Transporte,646,299,"10,110,481,422.00",646,0
8,Personeria de Barrancabermeja,606,262,"10,067,197,084.00",606,0
5,Hospital Regional del Magdalena Medio,399,220,"6,304,436,623.00",399,0
2,Contraloria de Barrancabermeja,323,124,"5,687,558,162.67",323,0


Servicios de persona natural en ESE Barrancabermeja: 1824


,anio_periodo,contratos,personas,valor_total
0,2022,50,45,"285,211,637.00"
1,2023,141,111,"1,490,855,641.33"
2,2024,292,263,"4,509,745,070.00"
3,2025,667,499,"5,473,322,360.00"
4,2026,674,487,"11,313,068,286.96"


In [53]:
# 32. Auditar subtipo CPS por alcalde

cps_alcaldia = cps.loc[
    cps["es_alcaldia_analisis"]
    & cps["alcalde"].isin(["Alfonso Eljach", "Jonathan Vasquez"])
].copy()

calidad_subtipo_alcalde = (
    cps_alcaldia.groupby("alcalde")
    .agg(
        contratos=("contrato_llave", "nunique"),
        subtipo_claro=("tipo_cps_claro", "sum")
    )
    .reset_index()
)

calidad_subtipo_alcalde["subtipo_no_claro"] = (
    calidad_subtipo_alcalde["contratos"]
    - calidad_subtipo_alcalde["subtipo_claro"]
)
calidad_subtipo_alcalde["porcentaje_claro"] = (
    calidad_subtipo_alcalde["subtipo_claro"]
    / calidad_subtipo_alcalde["contratos"]
    * 100
).round(2)

print(cps["tipo_cps"].value_counts())
display(calidad_subtipo_alcalde)

tipo_cps
Profesional claro                    15515
Apoyo a la gestion claro             14816
Justificacion general sin subtipo     2032
Mixto o ambiguo en descripcion         415
Name: count, dtype: int64


,alcalde,contratos,subtipo_claro,subtipo_no_claro,porcentaje_claro
0,Alfonso Eljach,12810,11519,1291,89.92
1,Jonathan Vasquez,13513,12661,852,93.69


In [54]:
# 33. Crear flags de valores mensuales extremos por entidad, subtipo y año

cps["flag_valor_mensual_extremo"] = False

cps["entidad_outlier"] = (
    cps["entidad_corta"]
    .fillna(cps["nombre_entidad"])
)

validos_outlier = cps.loc[
    cps["tipo_cps_claro"]
    & cps["valor_mensual_ajustado"].notna()
    & cps["anio_periodo"].notna()
].copy()

limites_outliers = (
    validos_outlier.groupby(
        ["entidad_outlier", "tipo_cps", "anio_periodo"],
        dropna=False
    )
    .agg(
        n=("valor_mensual_ajustado", "size"),
        limite_bajo=(
            "valor_mensual_ajustado",
            lambda x: x.quantile(0.005)
        ),
        limite_alto=(
            "valor_mensual_ajustado",
            lambda x: x.quantile(0.995)
        )
    )
    .reset_index()
)

validos_outlier = validos_outlier.merge(
    limites_outliers,
    on=["entidad_outlier", "tipo_cps", "anio_periodo"],
    how="left"
)

indices_extremos = validos_outlier.loc[
    (validos_outlier["n"] >= 30)
    & (
        validos_outlier["valor_mensual_ajustado"]
        .lt(validos_outlier["limite_bajo"])
        |
        validos_outlier["valor_mensual_ajustado"]
        .gt(validos_outlier["limite_alto"])
    ),
    "fila_origen"
]

cps.loc[
    cps["fila_origen"].isin(indices_extremos),
    "flag_valor_mensual_extremo"
] = True

print(
    "CPS con valor mensual extremo:",
    int(cps["flag_valor_mensual_extremo"].sum())
)

CPS con valor mensual extremo: 347


In [55]:
# 34. Auditar nulos CPS

resumen_nulos_cps = (
    cps.groupby("alcalde", dropna=False)
    .agg(
        contratos=("contrato_llave", "nunique"),
        sin_fecha_firma=("fecha_de_firma", lambda x: x.isna().sum()),
        sin_fecha_inicio=("fecha_de_inicio_del_contrato", lambda x: x.isna().sum()),
        sin_duracion=("duracion_dias", lambda x: x.isna().sum()),
        sin_valor=("valor_contrato_num", lambda x: x.isna().sum()),
        sin_valor_mensual=("valor_mensual_equivalente", lambda x: x.isna().sum()),
        subtipo_no_claro=("tipo_cps_claro", lambda x: (~x).sum())
    )
    .reset_index()
)

display(resumen_nulos_cps)
print("Política: no imputar fechas, valores ni subtipo CPS. Excluir solo de la métrica afectada.")

,alcalde,contratos,sin_fecha_firma,sin_fecha_inicio,sin_duracion,sin_valor,sin_valor_mensual,subtipo_no_claro
0,Alfonso Eljach,12810,0,72,72,0,88,1291
1,Jonathan Vasquez,13513,0,174,177,0,223,852
2,No aplica,6455,0,39,40,0,53,304


Política: no imputar fechas, valores ni subtipo CPS. Excluir solo de la métrica afectada.


In [56]:
# 35. Auditar valores en cero por familia

auditoria_valor_cero = (
    base_contratos.loc[base_contratos["flag_valor_cero"]]
    .groupby("familia_contrato", dropna=False)
    .agg(
        contratos=("contrato_llave", "nunique"),
        proveedores=("proveedor_llave", "nunique")
    )
    .reset_index()
    .sort_values("contratos", ascending=False)
)

display(auditoria_valor_cero)

,familia_contrato,contratos,proveedores
2,Comodato,61,48
3,Convenios e interadministrativos,7,6
5,Regimen especial persona juridica o grupo,6,4
0,Arrendamiento,3,1
1,CPS persona natural estricto,2,2
4,Entidad sin animo de lucro (ESAL) / Decreto 092,2,2
7,Servicios empresa o grupo,2,2
6,Seguros,1,1


In [57]:
# 36. Auditar fechas invertidas

revision_fechas_invertidas = base_contratos.loc[
    base_contratos["flag_fecha_invertida"],
    [
        "id_contrato",
        "referencia_del_contrato",
        "nombre_entidad",
        "proveedor_adjudicado",
        "fecha_de_inicio_del_contrato",
        "fecha_de_fin_del_contrato",
        "valor_contrato_num",
        "urlproceso"
    ]
].copy()

print(f"Fechas invertidas: {len(revision_fechas_invertidas):,}")
display(revision_fechas_invertidas)

Fechas invertidas: 5


,id_contrato,referencia_del_contrato,nombre_entidad,proveedor_adjudicado,fecha_de_inicio_del_contrato,fecha_de_fin_del_contrato,valor_contrato_num,urlproceso
22487,CO1.PCCNTR.6952691,4092-24,ALCALDIA DISTRITAL BARRANCABERMEJA,Jenny Marcela Rueda Jimenez,2025-06-25,2024-12-25,"9,000,000.00",{'url': 'https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.6951856&isFromPublicArea=True&isModal=true&asPopupView=true'}
30724,CO1.PCCNTR.8622471,HRMM-CD-417-2025,E.S.E HOSPITAL REGIONAL DEL MAGDALENA MEDIO//,Juan Esteban Vera Caballero,2025-11-26,2025-11-21,"5,066,645.00",{'url': 'https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.9172786&isFromPublicArea=True&isModal=true&asPopupView=true'}
30812,CO1.PCCNTR.8641533,CONTRATO 4628-25,ALCALDIA DISTRITAL BARRANCABERMEJA,CORPORACIÓN CREE,2025-11-27,2025-11-25,"454,800,000.00",{'url': 'https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.9158200&isFromPublicArea=True&isModal=true&asPopupView=true'}
30868,CO1.PCCNTR.8653182,CONTRATO 4957-25,ALCALDIA DISTRITAL BARRANCABERMEJA,ricardo enrique romero medina,2025-12-29,2025-12-28,"3,500,000.00",{'url': 'https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.9214858&isFromPublicArea=True&isModal=true&asPopupView=true'}
32398,CO1.PCCNTR.8909989,CONTRATO 1200-26,ALCALDIA DISTRITAL BARRANCABERMEJA,INGRID JOHANA JAIMES NUÑEZ,2026-01-19,2026-01-15,"7,500,000.00",{'url': 'https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.9543377&isFromPublicArea=True&isModal=true&asPopupView=true'}


In [58]:
# 36B. Auditar duraciones atipicas de CPS

# Dos problemas distintos, ambos de captura en SECOP:
#  a) Duracion cero: la fecha de fin no fue diligenciada.
#  b) CPS de mas de 12 meses: atipico para prestacion de servicios; suele ser
#     un error de digitacion en el ano de la fecha de fin.
# Politica del proyecto: se marcan y auditan, NUNCA se imputan ni se borran.

LIMITE_MESES_CPS = 12

base_contratos["flag_cps_duracion_atipica"] = (
    base_contratos["es_cps_estricto"]
    & base_contratos["duracion_meses_exacta"].gt(LIMITE_MESES_CPS)
)

base_contratos["flag_duracion_no_analizable"] = (
    base_contratos["flag_duracion_cero"]
    | base_contratos["flag_cps_duracion_atipica"]
)

revision_duracion_atipica = base_contratos.loc[
    base_contratos["flag_duracion_no_analizable"],
    [
        "id_contrato",
        "referencia_del_contrato",
        "entidad_analisis",
        "alcalde",
        "proveedor_adjudicado",
        "fecha_de_firma",
        "fecha_de_inicio_del_contrato",
        "fecha_de_fin_del_contrato",
        "duracion_dias",
        "duracion_meses_exacta",
        "valor_contrato_num",
        "valor_mensual_equivalente",
        "es_cps_estricto",
        "flag_duracion_cero",
        "flag_cps_duracion_atipica",
        "urlproceso"
    ]
].copy()

revision_duracion_atipica["motivo"] = np.where(
    revision_duracion_atipica["flag_duracion_cero"],
    "Duracion cero: fecha de fin no diligenciada",
    f"CPS de mas de {LIMITE_MESES_CPS} meses: probable error en el ano de la fecha de fin"
)

n_duracion_cero = int(base_contratos["flag_duracion_cero"].sum())
n_cps_duracion_atipica = int(base_contratos["flag_cps_duracion_atipica"].sum())

print(f"Contratos con duracion cero              : {n_duracion_cero}")
print(f"CPS con duracion mayor a {LIMITE_MESES_CPS} meses        : {n_cps_duracion_atipica}")
print(f"Total no analizables para duracion       : {len(revision_duracion_atipica)}")
print()
print("Estos registros se conservan en la base. Solo quedan excluidos de las")
print("estadisticas de duracion y de valor mensual, y se reportan para revision.")
display(revision_duracion_atipica.head(20))

Contratos con duracion cero              : 107
CPS con duracion mayor a 12 meses        : 1
Total no analizables para duracion       : 108

Estos registros se conservan en la base. Solo quedan excluidos de las
estadisticas de duracion y de valor mensual, y se reportan para revision.


,id_contrato,referencia_del_contrato,entidad_analisis,alcalde,proveedor_adjudicado,fecha_de_firma,fecha_de_inicio_del_contrato,fecha_de_fin_del_contrato,duracion_dias,duracion_meses_exacta,valor_contrato_num,valor_mensual_equivalente,es_cps_estricto,flag_duracion_cero,flag_cps_duracion_atipica,urlproceso,motivo
1395,CO1.PCCNTR.2745744,CONTRATO 1911-21,Alcaldia Distrital de Barrancabermeja,Alfonso Eljach,priscila paramo gonzalez,2021-08-09,2021-08-10,2021-08-10,0.00,0.00,"9,900,000.00",NaN,True,True,False,{'url': 'https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.2160791&isFromPublicArea=True&isModal=true&asPopupView=true'},Duracion cero: fecha de fin no diligenciada
1725,CO1.PCCNTR.2794609,CONTRATO 2218-21,Alcaldia Distrital de Barrancabermeja,Alfonso Eljach,Johanna Marcela Molina Santos,2021-08-25,2021-10-11,2021-10-11,0.00,0.00,"13,650,000.00",NaN,True,True,False,{'url': 'https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.2200022&isFromPublicArea=True&isModal=true&asPopupView=true'},Duracion cero: fecha de fin no diligenciada
3406,CO1.PCCNTR.3071245,CONTRATO 3666-21,Alcaldia Distrital de Barrancabermeja,Alfonso Eljach,Javier Mauricio Devera Bastidas,2021-11-29,2021-12-30,2021-12-30,0.00,0.00,"2,000,000.00",NaN,True,True,False,{'url': 'https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.2421924&isFromPublicArea=True&isModal=true&asPopupView=true'},Duracion cero: fecha de fin no diligenciada
3585,CO1.PCCNTR.3149659,CONTRATO 3800-21,Alcaldia Distrital de Barrancabermeja,Alfonso Eljach,Emperatriz diaz,2021-12-29,2021-12-31,2021-12-31,0.00,0.00,"149,992,224.00",NaN,False,True,False,{'url': 'https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.2448193&isFromPublicArea=True&isModal=true&asPopupView=true'},Duracion cero: fecha de fin no diligenciada
4597,CO1.PCCNTR.3368191,0678-22,Alcaldia Distrital de Barrancabermeja,Alfonso Eljach,YANITH RUEDA NAVARRO,2022-01-23,2022-01-23,2022-01-23,0.00,0.00,"12,000,000.00",NaN,True,True,False,{'url': 'https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.2668133&isFromPublicArea=True&isModal=true&asPopupView=true'},Duracion cero: fecha de fin no diligenciada
6202,CO1.PCCNTR.3520254,CONTRATO 2046-22,Alcaldia Distrital de Barrancabermeja,Alfonso Eljach,LINO MANUEL PERALTA SOLAR,2022-01-28,2022-01-29,2022-01-29,0.00,0.00,"13,200,000.00",NaN,True,True,False,{'url': 'https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.2789567&isFromPublicArea=True&isModal=true&asPopupView=true'},Duracion cero: fecha de fin no diligenciada
6470,CO1.PCCNTR.3549234,CONTRATO 3243-22,Alcaldia Distrital de Barrancabermeja,Alfonso Eljach,Yeimi Andrea Ferreira Meneses,2022-02-02,2022-02-03,2022-02-03,0.00,0.00,"6,400,000.00",NaN,True,True,False,{'url': 'https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.2811178&isFromPublicArea=True&isModal=true&asPopupView=true'},Duracion cero: fecha de fin no diligenciada
6745,CO1.PCCNTR.3556020,CONTRATO 2901-22,Alcaldia Distrital de Barrancabermeja,Alfonso Eljach,ALEXANDER SAID MEJIA GOMEZ,2022-02-03,2022-02-03,2022-02-03,0.00,0.00,"3,600,000.00",NaN,True,True,False,{'url': 'https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.2817652&isFromPublicArea=True&isModal=true&asPopupView=true'},Duracion cero: fecha de fin no diligenciada
9683,CO1.PCCNTR.4098600,CONTRATO 5508-22,Alcaldia Distrital de Barrancabermeja,Alfonso Eljach,Oscar Fernando Romero Comas,2022-10-06,2022-10-28,2022-10-28,0.00,0.00,"11,050,000.00",NaN,True,True,False,{'url': 'https://community.secop.gov.co/Public/Tendering/OpportunityDetail/Index?noticeUID=CO1.NTC.3370676&isFromPublicArea=True&isModal=true&asPopupView=true'},Duracion cero: fecha de fin no diligenciada
10302,CO1.PCCNTR.4209218,CTO-454-2022,Instituto para el Fomen

In [59]:
# 36C. Auditar contratos con ejecucion parcial (terminacion anticipada)

# Estos contratos son la causa principal de valores mensuales inflados.
# Se auditan aparte para poder verificarlos contra el expediente de SECOP.

revision_ejecucion_parcial = base_contratos.loc[
    base_contratos["flag_ejecucion_parcial"] & base_contratos["es_cps_estricto"],
    [
        "id_contrato",
        "referencia_del_contrato",
        "alcalde",
        "proveedor_adjudicado",
        "fecha_de_inicio_del_contrato",
        "fecha_de_fin_del_contrato",
        "duracion_meses_exacta",
        "valor_contrato_num",
        "valor_ejecutado_num",
        "pct_ejecucion",
        "valor_mensual_equivalente",
        "valor_mensual_ajustado",
        "estado_contrato",
        "urlproceso"
    ]
].copy()

revision_ejecucion_parcial["sobreestimacion_veces"] = (
    revision_ejecucion_parcial["valor_mensual_equivalente"]
    / revision_ejecucion_parcial["valor_mensual_ajustado"]
).round(2)

resumen_ejecucion_parcial = (
    base_contratos.loc[base_contratos["es_cps_estricto"]]
    .groupby(["alcalde", "fuente_valor_mensual"], dropna=False)
    .agg(contratos=("contrato_llave", "nunique"))
    .reset_index()
)

n_ejecucion_parcial = len(revision_ejecucion_parcial)
n_sin_base_confiable = int(
    (
        base_contratos["flag_valor_mensual_no_confiable"]
        & base_contratos["es_cps_estricto"]
    ).sum()
)

print(f"CPS con ejecucion parcial (valor mensual ajustado): {n_ejecucion_parcial:,}")
print(f"CPS sin base confiable para valor mensual         : {n_sin_base_confiable:,}")
print()
if n_ejecucion_parcial:
    print("Sobreestimacion que se habria producido sin el ajuste:")
    print(
        revision_ejecucion_parcial["sobreestimacion_veces"]
        .describe(percentiles=[0.5, 0.9, 0.99]).round(2).to_string()
    )
display(resumen_ejecucion_parcial)
display(
    revision_ejecucion_parcial
    .nlargest(15, "sobreestimacion_veces")
    [["referencia_del_contrato", "proveedor_adjudicado", "valor_contrato_num",
      "valor_ejecutado_num", "duracion_meses_exacta",
      "valor_mensual_equivalente", "valor_mensual_ajustado", "sobreestimacion_veces"]]
)

CPS con ejecucion parcial (valor mensual ajustado): 1,171
CPS sin base confiable para valor mensual         : 322

Sobreestimacion que se habria producido sin el ajuste:
count       1,171.00
mean        2,563.56
std        50,571.98
min             1.05
50%             1.25
90%             2.48
99%             7.65
max     1,000,000.00


,alcalde,fuente_valor_mensual,contratos
0,Alfonso Eljach,Sin base confiable,77
1,Alfonso Eljach,Valor del contrato,11982
2,Alfonso Eljach,Valor ejecutado (terminacion anticipada),751
3,Jonathan Vasquez,Sin base confiable,72
4,Jonathan Vasquez,Valor del contrato,13147
5,Jonathan Vasquez,Valor ejecutado (terminacion anticipada),294
6,No aplica,Sin base confiable,173
7,No aplica,Valor del contrato,6156
8,No aplica,Valor ejecutado (terminacion anticipada),126


,referencia_del_contrato,proveedor_adjudicado,valor_contrato_num,valor_ejecutado_num,duracion_meses_exacta,valor_mensual_equivalente,valor_mensual_ajustado,sobreestimacion_veces
233,CONTRATO 0839-21,Olga Sofia Galvis Miranda,"6,000,000.00",6,2.99,"2,007,032.97",2.01,"1,000,000.00"
920,CONTRATO 1501-21,LIBARDO CASTILLO ARDILA,"6,000,000.00",6,2.99,"2,007,032.97",2.01,"1,000,000.00"
3393,CONTRATO 3640-21,sandra milena caballero gomez,"2,000,000.00",2,0.95,"2,099,310.34",2.10,"1,000,000.00"
7195,CONTRATO 3458-22,XIMENA MESA VERGARA,"10,000,000.00",583333,3.98,"2,515,702.48","146,749.23",17.14
24425,031-8-2025,MARCELA HERNANDEZ FLOREZ,"19,000,000.00",1140000,4.89,"3,881,610.74","232,896.64",16.67
13520,CONTRATO 103 DE 2023,DIANA MILENA CAMARGO GONZALEZ,"39,000,000.00",3000000,6.47,"6,026,192.89","463,553.30",13.00
29955,CD-CMB-273-25,LEDYS JOHANA RAMOS FLOREZ,"6,529,501.00",502269,1.28,"5,096,359.24","392,027.39",13.00
24721,CONTRATO 0827-25,MARIEL SARAI ROSAS VELOZA,"14,400,000.00",1200000,3.98,"3,622,611.57","301,884.30",12.00
3386,CONTRATO 3651-21,Claudia Katherine Torres Valencia,"2,000,000.00",200000,0.95,"2,099,310.34","209,931.03",10.00
13199,CTO-190-2023,FELIX DAVID PARDO BARRETO,"13,200,000.00",1613333,6.01,"2,195,672.13","268,359.87",8.18


In [60]:
# 37. Auditar días adicionados por alcalde y subtipo

cps_adiciones = cps.loc[
    cps["es_alcaldia_analisis"]
    & cps["alcalde"].isin(["Alfonso Eljach", "Jonathan Vasquez"])
].copy()

resumen_adiciones_cps = (
    cps_adiciones.groupby(
        ["alcalde", "tipo_cps"],
        dropna=False
    )
    .agg(
        contratos=("contrato_llave", "nunique"),
        contratos_con_adicion=(
            "flag_tiene_dias_adicionados",
            "sum"
        ),
        dias_adicionados_mediana=(
            "dias_adicionados_num",
            lambda x: x[x.gt(0)].median()
        ),
        dias_adicionados_max=(
            "dias_adicionados_num",
            "max"
        )
    )
    .reset_index()
)

resumen_adiciones_cps["porcentaje_con_adicion"] = (
    resumen_adiciones_cps["contratos_con_adicion"]
    / resumen_adiciones_cps["contratos"]
    * 100
).round(2)

display(
    resumen_adiciones_cps.sort_values(
        ["alcalde", "tipo_cps"]
    )
)

,alcalde,tipo_cps,contratos,contratos_con_adicion,dias_adicionados_mediana,dias_adicionados_max,porcentaje_con_adicion
0,Alfonso Eljach,Apoyo a la gestion claro,5928,178,30.00,100,3.00
1,Alfonso Eljach,Justificacion general sin subtipo,1023,87,29.00,65,8.50
2,Alfonso Eljach,Mixto o ambiguo en descripcion,268,15,59.00,61,5.60
3,Alfonso Eljach,Profesional claro,5591,323,34.00,146,5.78
4,Jonathan Vasquez,Apoyo a la gestion claro,6221,745,31.00,385,11.98
5,Jonathan Vasquez,Justificacion general sin subtipo,730,59,40.00,100,8.08
6,Jonathan Vasquez,Mixto o ambiguo en descripcion,122,12,60.00,61,9.84
7,Jonathan Vasquez,Profesional claro,6440,1080,32.00,169,16.77


In [61]:
# 38. Auditar Entidades sin animo de lucro (ESAL)

auditoria_esal = (
    base_analitica.loc[
        base_analitica["familia_contrato"].eq(
            "Entidad sin animo de lucro (ESAL) / Decreto 092"
        )
    ]
    .groupby(
        [
            "es_decreto_092",
            "es_convenio_asociacion",
            "menciona_esal",
            "tipo_de_contrato",
            "modalidad_de_contratacion",
            "tipo_proveedor"
        ],
        dropna=False
    )
    .agg(
        contratos=("contrato_llave", "nunique"),
        proveedores=("proveedor_llave", "nunique"),
        valor_total=("valor_contrato_num", "sum")
    )
    .reset_index()
    .sort_values("valor_total", ascending=False)
)

print(
    "Contratos confirmados con ESAL:",
    int(
        base_analitica["familia_contrato"]
        .eq("Entidad sin animo de lucro (ESAL) / Decreto 092")
        .sum()
    )
)

display(auditoria_esal.head(50))

Contratos confirmados con ESAL: 915


,es_decreto_092,es_convenio_asociacion,menciona_esal,tipo_de_contrato,modalidad_de_contratacion,tipo_proveedor,contratos,proveedores,valor_total
0,True,False,False,Decreto 092 de 2017,Contratación régimen especial,Persona juridica,502,121,"164,499,689,907.28"
2,True,False,False,Decreto 092 de 2017,Contratación régimen especial (con ofertas),Persona juridica,409,72,"93,528,404,389.82"
1,True,False,False,Decreto 092 de 2017,Contratación régimen especial (con ofertas),Grupo o consorcio,4,4,"5,267,662,516.00"


In [62]:
# 39. Auditar servicios de persona natural detectados por objeto contractual

revision_servicios_por_objeto = (
    base_analitica.loc[
        base_analitica["familia_contrato"].eq(
            "Prestacion servicios persona natural por objeto explicito"
        ),
        [
            "id_contrato",
            "referencia_del_contrato",
            "entidad_analisis",
            "nombre_entidad",
            "tipo_de_contrato",
            "modalidad_de_contratacion",
            "justificacion_modalidad_de",
            "descripcion_del_proceso",
            "proveedor_adjudicado",
            "documento_proveedor",
            "valor_contrato_num",
            "fecha_de_firma",
            "fecha_de_inicio_del_contrato",
            "fecha_de_fin_del_contrato",
            "urlproceso"
        ]
    ]
    .copy()
)

resumen_servicios_por_objeto_entidad = (
    revision_servicios_por_objeto
    .groupby(
        "entidad_analisis",
        dropna=False
    )
    .agg(
        contratos=("id_contrato", "nunique"),
        personas=("documento_proveedor", "nunique"),
        valor_total=("valor_contrato_num", "sum")
    )
    .reset_index()
    .sort_values(
        "contratos",
        ascending=False
    )
)

print(
    "Servicios de persona natural detectados por objeto:",
    len(revision_servicios_por_objeto)
)

display(resumen_servicios_por_objeto_entidad)

# ESE Barrancabermeja se mantiene como entidad separada de la Alcaldia
revision_servicios_ese = (
    revision_servicios_por_objeto.loc[
        revision_servicios_por_objeto[
            "entidad_analisis"
        ].eq(
            "Empresa Social del Estado Barrancabermeja"
        )
    ]
    .copy()
)

print(
    "Servicios detectados por objeto en ESE Barrancabermeja:",
    len(revision_servicios_ese)
)

display(
    revision_servicios_ese[
        [
            "referencia_del_contrato",
            "descripcion_del_proceso",
            "proveedor_adjudicado",
            "valor_contrato_num"
        ]
    ].head(30)
)

Servicios de persona natural detectados por objeto: 1827


,entidad_analisis,contratos,personas,valor_total
1,Empresa Social del Estado Barrancabermeja,1824,1024,"23,072,202,995.29"
0,Alcaldia Distrital de Barrancabermeja,1,1,"8,400,000.00"
2,Empresa de Desarrollo Urbano y Vivienda,1,1,"24,026,000.00"
3,Instituto para el Fomento del Deporte y la Recreacion,1,1,"1,015,000.00"


Servicios detectados por objeto en ESE Barrancabermeja: 1824


,referencia_del_contrato,descripcion_del_proceso,proveedor_adjudicado,valor_contrato_num
1995,26-00749,PRESTACIÓN DE SERVICIOS PROFESIONALES COMO MEDICO EN LOS EQUIPOS BÁSICOS EN SALUD (EBS) PARA EL APOYO A LA E.S.E BARRANCABERMEJA; EN EL FORTALECIMIENTO Y ATENCIÓN DEL NIVEL PRI...,JOSE FERNANDO SILVA LLANOS,"34,200,000.00"
3492,26-00747,PRESTACION DE SERVICIOS PARA EL LAVADO DE TANQUES ELEVADOS Y CISTERNAS DE LOS CENTROS DE SALUD DE LA ESE BARRANCABERMEJA.,RIBON CAMPO,"10,000,000.00"
7289,26-00748,PRESTACIÓN DE SERVICIOS PROFESIONALES COMO PSICOLOGO EN LOS EQUIPOS BÁSICOS EN SALUD (EBS) PARA EL APOYO A LA E.S.E BARRANCABERMEJA; EN EL FORTALECIMIENTO Y ATENCIÓN DEL NIVEL ...,DEICY URIBE IRREÑO,"25,200,000.00"
7303,22-00095,PRESTACION DE SERVICIOS PROFESIONALES UNIVESITARIOS COMO PROFESIONAL EN SALUD OCUPACIONAL; PARA BRINDAR APOYO AL CUMPLIMIENTO DEL CONTRATO INTERADMINISTRATIVO No. 3331-22 DEL P...,LESLYE TATIANA PALACIO,"7,062,000.00"
7324,22-00096,PRESTACIÓN DE SERVICIOS PROFESIONALES ESPECIALIZADOS COMO INGENIERO CIVIL PARA BRINDAR APOYO A LOS DIFERENTES PROYECTOS Y CONTRATOS DE OBRA QUE ADELANTE Y SUSCRIBA LA E.S.E BAR...,LUIS FERNANDO FONSECA VALENCIA,"8,000,000.00"
7379,22-00098,PRESTACION DE SERVICIOS PERSONALES COMO AUXILIAR ADMINISTRATIVO; PARA BRINDAR APOYO EN EL INVENTARIO QUE REALIZA ALMACEN DE LA E.S.E BARRANCABERMEJA,Clara Isabel Herrera jimenez,"1,264,740.00"
7421,22-00099,PRESTACION DE SERVICIOS PROFESIONALES ESPECIALIZADOS PARA LA LECTURA DE MUESTRAS DE CITOLOGIA CERVICO-VAGINAL; TOMADA Y ORDENADA A LAS USUARIAS DE BAJO NIVEL DE COMPLEJIDAD ATE...,DRA. YUDAMIS MARTINEZ NIEVES,"40,000,000.00"
7854,22-00107,PRESTACION DE SERVICIOS PROFESIONALES COMO PROFESIONAL EN EL ÁREA DE LA SALUD; PARA BRINDAR APOYO AL CUMPLIMIENTO DEL CONTRATO INTERADMINISTRATIVO No. 3331-22 DEL PLAN DE INTE...,KATHERINE ALVAREZ MEZA,"8,827,500.00"
7856,22-00106,PRESTACION DE SERVICIOS PROFESIONALES COMO ENFERMERA; PARA BRINDAR APOYO AL CUMPLIMIENTO DEL CONTRATO INTERADMINISTRATIVO No. 3331-22 DEL PLAN DE INTERVENCIONES COLECTIVAS; SU...,SANDRA LILIANA GOMEZ CEBALLOS,"7,468,065.00"
7988,22-00110,PRESTACION DE SERVICIOS PROFESIONALES ESPECIALIZADOS PARA LA ASESORIA Y APOYO DE LA SUBDIRECCION ADMINISTRATIVA Y FINANCIERA,SANDRA MILENA RAMIREZ BARROSO,"4,708,000.00"


In [63]:
# 40. Auditar régimen especial con persona jurídica o grupo

auditoria_regimen_especial_juridico = (
    base_analitica.loc[
        base_analitica["familia_contrato"].eq(
            "Regimen especial persona juridica o grupo"
        )
    ]
    .groupby(
        [
            "tipo_de_contrato",
            "modalidad_de_contratacion",
            "tipo_proveedor"
        ],
        dropna=False
    )
    .agg(
        contratos=("contrato_llave", "nunique"),
        proveedores=("proveedor_llave", "nunique"),
        valor_total=("valor_contrato_num", "sum")
    )
    .reset_index()
    .sort_values("valor_total", ascending=False)
)

print(
    "Contratos de régimen especial con persona jurídica o grupo:",
    int(base_analitica["familia_contrato"].eq(
        "Regimen especial persona juridica o grupo"
    ).sum())
)
display(auditoria_regimen_especial_juridico)

Contratos de régimen especial con persona jurídica o grupo: 116


,tipo_de_contrato,modalidad_de_contratacion,tipo_proveedor,contratos,proveedores,valor_total
0,Otro,Contratación régimen especial,Persona juridica,113,29,"127,274,906,097.63"
1,Otro,Contratación régimen especial (con ofertas),Grupo o consorcio,2,2,"6,635,802,276.40"
2,Otro,Contratación régimen especial (con ofertas),Persona juridica,1,1,"200,000,000.00"


In [64]:
# 41. Revisar familias contractuales válidas

resumen_familias = (
    base_analitica.groupby("familia_contrato", dropna=False)
    .agg(
        contratos=("contrato_llave", "nunique"),
        proveedores=("proveedor_llave", "nunique"),
        valor_total=("valor_contrato_num", "sum")
    )
    .reset_index()
    .sort_values("valor_total", ascending=False)
)

display(resumen_familias)

,familia_contrato,contratos,proveedores,valor_total
15,Servicios empresa o grupo,732,260,"522,930,177,346.63"
3,CPS persona natural estricto,32778,9926,"392,266,631,156.97"
8,Obra,97,75,"271,742,091,652.33"
7,Entidad sin animo de lucro (ESAL) / Decreto 092,915,181,"263,295,756,813.10"
6,Convenios e interadministrativos,83,31,"183,435,926,109.53"
12,Regimen especial persona juridica o grupo,116,32,"134,110,708,374.03"
2,Bienes y suministros,387,139,"88,376,566,033.97"
5,Consultoria e interventoria,94,63,"46,724,857,774.56"
10,Prestacion servicios persona natural por objeto explicito,1827,1027,"23,105,643,995.29"
0,Arrendamiento,173,66,"21,321,182,091.38"


In [65]:
# 42. Auditar tipo de contrato SECOP frente a familia analitica

auditoria_tipo_familia = (
    base_analitica.groupby(
        [
            "tipo_de_contrato",
            "familia_contrato"
        ],
        dropna=False
    )
    .agg(
        contratos=("contrato_llave", "nunique"),
        proveedores=("proveedor_llave", "nunique"),
        valor_total=("valor_contrato_num", "sum")
    )
    .reset_index()
    .sort_values(
        ["tipo_de_contrato", "contratos"],
        ascending=[True, False]
    )
)

display(auditoria_tipo_familia.head(100))

,tipo_de_contrato,familia_contrato,contratos,proveedores,valor_total
0,Arrendamiento de inmuebles,Arrendamiento,172,65,"20,421,513,292.38"
1,Arrendamiento de inmuebles,Prestacion servicios persona natural por objeto explicito,1,1,"1,015,000.00"
2,Arrendamiento de muebles,Arrendamiento,1,1,"899,668,799.00"
3,Asociación Público Privada,Asociacion publico privada,2,2,"419,146,317.62"
4,Comodato,Comodato,63,45,"1,583,770,428.00"
5,Compraventa,Bienes y suministros,169,84,"29,152,719,533.82"
7,Compraventa,Regimen especial persona natural por revisar,5,3,"175,394,800.00"
6,Compraventa,Convenios e interadministrativos,1,1,"886,550,000.00"
8,Consultoría,Consultoria e interventoria,35,22,"19,892,795,310.05"
10,Decreto 092 de 2017,Prestacion servicios persona natural por objeto explicito,1824,1024,"23,072,202,995.29"


In [66]:
# 43. Revisar duración de CPS claros en 24 meses comparables

cps_24 = cps_alcaldia.loc[
    cps_alcaldia["periodo_comparable_24m"]
    & cps_alcaldia["tipo_cps_claro"]
    & cps_alcaldia["apto_analisis_duracion"]
].copy()

def grupo_duracion_grafica(valor):
    if pd.isna(valor):
        return "Sin dato"
    valor = int(valor)
    if valor <= 0:
        return "<1 mes"
    if 1 <= valor <= 8:
        return f"{valor} mes" if valor == 1 else f"{valor} meses"
    if 9 <= valor <= 12:
        return "9 a 12 meses"
    return ">12 meses"

cps_24["duracion_grupo_grafica"] = (
    cps_24["duracion_meses_aprox"]
    .map(grupo_duracion_grafica)
)

ORDEN_DURACION = [
    "<1 mes",
    "1 mes",
    "2 meses",
    "3 meses",
    "4 meses",
    "5 meses",
    "6 meses",
    "7 meses",
    "8 meses",
    "9 a 12 meses",
    ">12 meses"
]

cps_24["duracion_grupo_grafica"] = pd.Categorical(
    cps_24["duracion_grupo_grafica"],
    categories=ORDEN_DURACION,
    ordered=True
)

resumen_duracion_24m = (
    cps_24.groupby(
        [
            "alcalde",
            "tipo_cps",
            "duracion_grupo_grafica"
        ],
        observed=True
    )
    .agg(
        contratos=("contrato_llave", "nunique")
    )
    .reset_index()
)

resumen_duracion_24m["porcentaje"] = (
    resumen_duracion_24m["contratos"]
    / resumen_duracion_24m.groupby(
        ["alcalde", "tipo_cps"]
    )["contratos"].transform("sum")
    * 100
).round(2)

display(resumen_duracion_24m)

,alcalde,tipo_cps,duracion_grupo_grafica,contratos,porcentaje
0,Alfonso Eljach,Apoyo a la gestion claro,<1 mes,69,1.48
1,Alfonso Eljach,Apoyo a la gestion claro,1 mes,353,7.59
2,Alfonso Eljach,Apoyo a la gestion claro,2 meses,437,9.40
3,Alfonso Eljach,Apoyo a la gestion claro,3 meses,1136,24.44
4,Alfonso Eljach,Apoyo a la gestion claro,4 meses,1424,30.63
5,Alfonso Eljach,Apoyo a la gestion claro,5 meses,686,14.76
6,Alfonso Eljach,Apoyo a la gestion claro,6 meses,410,8.82
7,Alfonso Eljach,Apoyo a la gestion claro,7 meses,66,1.42
8,Alfonso Eljach,Apoyo a la gestion claro,8 meses,29,0.62
9,Alfonso Eljach,Apoyo a la gestion claro,9 a 12 meses,39,0.84


In [67]:
# 44. Resumir CPS por subtipo claro

def resumen_por_subtipo(tabla):
    return (
        tabla.loc[
            tabla["tipo_cps_claro"]
            & tabla["apto_analisis_duracion"]
        ]
        .groupby(
            ["alcalde", "tipo_cps"]
        )
        .agg(
            contratos_cps=("contrato_llave", "nunique"),
            personas=("proveedor_llave", "nunique"),
            duracion_mediana_meses=(
                "duracion_meses_exacta",
                "median"
            ),
            duracion_p25_meses=(
                "duracion_meses_exacta",
                lambda x: x.quantile(0.25)
            ),
            duracion_p75_meses=(
                "duracion_meses_exacta",
                lambda x: x.quantile(0.75)
            ),
            valor_mensual_mediano_nominal=(
                "valor_mensual_equivalente",
                "median"
            )
        )
        .reset_index()
    )

resumen_cps_subtipo_observado = resumen_por_subtipo(
    cps_alcaldia
)

resumen_cps_subtipo_24m = resumen_por_subtipo(
    cps_alcaldia.loc[
        cps_alcaldia["periodo_comparable_24m"]
    ]
)

print("Periodo observado")
display(resumen_cps_subtipo_observado)

print("Ventana comparable de 24 meses")
display(resumen_cps_subtipo_24m)

print(
    "Valores monetarios nominales. "
    "No comparar poder adquisitivo sin IPC."
)

Periodo observado


,alcalde,tipo_cps,contratos_cps,personas,duracion_mediana_meses,duracion_p25_meses,duracion_p75_meses,valor_mensual_mediano_nominal
0,Alfonso Eljach,Apoyo a la gestion claro,5879,2736,3.75,2.89,4.01,"2,012,561.98"
1,Alfonso Eljach,Profesional claro,5559,2084,3.91,2.89,4.96,"3,551,333.33"
2,Jonathan Vasquez,Apoyo a la gestion claro,6124,3106,2.92,2.23,3.98,"2,565,168.54"
3,Jonathan Vasquez,Profesional claro,6327,2162,3.91,2.51,4.83,"4,058,666.67"


Ventana comparable de 24 meses


,alcalde,tipo_cps,contratos_cps,personas,duracion_mediana_meses,duracion_p25_meses,duracion_p75_meses,valor_mensual_mediano_nominal
0,Alfonso Eljach,Apoyo a la gestion claro,4649,2489,3.91,2.96,4.89,"2,007,032.97"
1,Alfonso Eljach,Profesional claro,4280,1876,3.91,2.96,5.49,"3,551,333.33"
2,Jonathan Vasquez,Apoyo a la gestion claro,3989,2241,2.96,1.97,3.91,"2,536,666.67"
3,Jonathan Vasquez,Profesional claro,4545,1875,2.99,1.97,4.01,"4,058,666.67"


Valores monetarios nominales. No comparar poder adquisitivo sin IPC.


In [68]:
# 45. Crear resúmenes de ventanas comparables

def resumen_ventana(df_cps, columna_ventana):
    temp = df_cps.loc[df_cps[columna_ventana]].copy()
    return (
        temp.groupby("alcalde")
        .agg(
            contratos_cps=("contrato_llave", "nunique"),
            personas=("proveedor_llave", "nunique"),
            duracion_mediana_meses=("duracion_meses_exacta", "median"),
            contratos_por_persona=("proveedor_llave", lambda x: len(x) / x.nunique())
        )
        .reset_index()
    )

resumen_32m = resumen_ventana(cps_alcaldia, "periodo_comparable_32m")
resumen_24m = resumen_ventana(cps_alcaldia, "periodo_comparable_24m")
resumen_anio3 = resumen_ventana(cps_alcaldia, "periodo_anio3_mismo_corte")

print("Ventana 32 meses")
display(resumen_32m)
print("Ventana 24 meses completos")
display(resumen_24m)
print("Tercer año al mismo corte: 1 de enero a 6 de septiembre")
display(resumen_anio3)

Ventana 32 meses


,alcalde,contratos_cps,personas,duracion_mediana_meses,contratos_por_persona
0,Alfonso Eljach,12558,4971,3.91,2.53
1,Jonathan Vasquez,13470,5312,2.99,2.54


Ventana 24 meses completos


,alcalde,contratos_cps,personas,duracion_mediana_meses,contratos_por_persona
0,Alfonso Eljach,9970,4591,3.91,2.17
1,Jonathan Vasquez,9270,4236,2.96,2.19


Tercer año al mismo corte: 1 de enero a 6 de septiembre


,alcalde,contratos_cps,personas,duracion_mediana_meses,contratos_por_persona
0,Alfonso Eljach,3593,2858,3.98,1.26
1,Jonathan Vasquez,4243,3192,3.91,1.33


In [69]:
# 46. Medir recurrencia de personas en 24 meses comparables

cps_24_personas = (
    cps_alcaldia.loc[
        cps_alcaldia["periodo_comparable_24m"]
    ]
    .groupby(
        ["alcalde", "proveedor_llave"],
        dropna=False
    )
    .agg(
        contratos=("contrato_llave", "nunique"),
        primer_contrato=("fecha_de_firma", "min"),
        ultimo_contrato=("fecha_de_firma", "max")
    )
    .reset_index()
)

def categoria_contratos_persona(n):
    if n == 1:
        return "1 contrato"
    if n == 2:
        return "2 contratos"
    if n == 3:
        return "3 contratos"
    if n == 4:
        return "4 contratos"
    return "5 o mas"

cps_24_personas["categoria_contratos"] = (
    cps_24_personas["contratos"]
    .map(categoria_contratos_persona)
)

ORDEN_RECURRENCIA = [
    "1 contrato",
    "2 contratos",
    "3 contratos",
    "4 contratos",
    "5 o mas"
]

cps_24_personas["categoria_contratos"] = pd.Categorical(
    cps_24_personas["categoria_contratos"],
    categories=ORDEN_RECURRENCIA,
    ordered=True
)

distribucion_recurrencia_24m = (
    cps_24_personas.groupby(
        ["alcalde", "categoria_contratos"],
        observed=True
    )
    .agg(
        personas=("proveedor_llave", "nunique")
    )
    .reset_index()
)

distribucion_recurrencia_24m["porcentaje_personas"] = (
    distribucion_recurrencia_24m["personas"]
    / distribucion_recurrencia_24m.groupby(
        "alcalde"
    )["personas"].transform("sum")
    * 100
).round(2)

resumen_recurrencia_24m = (
    cps_24_personas.groupby("alcalde")
    .agg(
        personas=("proveedor_llave", "nunique"),
        contratos=("contratos", "sum"),
        contratos_mediana=("contratos", "median"),
        contratos_p90=(
            "contratos",
            lambda x: x.quantile(0.90)
        ),
        contratos_max=("contratos", "max"),
        personas_2_o_mas=(
            "contratos",
            lambda x: (x >= 2).sum()
        ),
        personas_3_o_mas=(
            "contratos",
            lambda x: (x >= 3).sum()
        )
    )
    .reset_index()
)

resumen_recurrencia_24m["porcentaje_2_o_mas"] = (
    resumen_recurrencia_24m["personas_2_o_mas"]
    / resumen_recurrencia_24m["personas"]
    * 100
).round(2)

resumen_recurrencia_24m["porcentaje_3_o_mas"] = (
    resumen_recurrencia_24m["personas_3_o_mas"]
    / resumen_recurrencia_24m["personas"]
    * 100
).round(2)

display(resumen_recurrencia_24m)
display(distribucion_recurrencia_24m)

,alcalde,personas,contratos,contratos_mediana,contratos_p90,contratos_max,personas_2_o_mas,personas_3_o_mas,porcentaje_2_o_mas,porcentaje_3_o_mas
0,Alfonso Eljach,4591,9970,2.00,4.00,9,2624,1647,57.16,35.87
1,Jonathan Vasquez,4236,9270,2.00,4.00,9,2200,1425,51.94,33.64


,alcalde,categoria_contratos,personas,porcentaje_personas
0,Alfonso Eljach,1 contrato,1967,42.84
1,Alfonso Eljach,2 contratos,977,21.28
2,Alfonso Eljach,3 contratos,824,17.95
3,Alfonso Eljach,4 contratos,583,12.70
4,Alfonso Eljach,5 o mas,240,5.23
5,Jonathan Vasquez,1 contrato,2036,48.06
6,Jonathan Vasquez,2 contratos,775,18.30
7,Jonathan Vasquez,3 contratos,493,11.64
8,Jonathan Vasquez,4 contratos,575,13.57
9,Jonathan Vasquez,5 o mas,357,8.43


In [70]:
# 47. Crear semaforo de calidad

n_ids_duplicados = (
    ids_repetidos["id_contrato_norm"]
    .nunique()
)

n_proveedor_revisar = int(
    base_contratos["tipo_proveedor"]
    .eq("Por revisar")
    .sum()
)

n_estados_revisar = len(
    registros_estado_revisar
)

n_fechas_invertidas = int(
    base_contratos["flag_fecha_invertida"]
    .sum()
)

n_otros_revisar = int(
    base_analitica["familia_contrato"]
    .eq("Otros por revisar")
    .sum()
)

n_regimen_natural_revisar = int(
    base_analitica["familia_contrato"]
    .eq(
        "Regimen especial persona natural por revisar"
    )
    .sum()
)

n_cps_probables = int(
    (
        cps_ampliado["es_cps_estricto"] == False
    ).sum()
)

n_cps_fuera_familia = len(
    integridad_cps_familia
)

n_documentos_sospechosos = int(
    base_contratos["flag_documento_sospechoso"]
    .sum()
)

n_inconsistencia_alcaldia = int(
    base_contratos["flag_inconsistencia_alcaldia"]
    .sum()
)

n_esal_con_persona_natural = int(
    (
        base_analitica["familia_contrato"]
        .eq(
            "Entidad sin animo de lucro (ESAL) / Decreto 092"
        )
        & base_analitica["tipo_proveedor"]
        .eq("Persona natural")
    ).sum()
)

n_ese_en_cps_alcaldia = int(
    (
        cps_alcaldia["entidad_analisis"]
        .eq(
            "Empresa Social del Estado Barrancabermeja"
        )
    ).sum()
)

min_claridad = (
    calidad_subtipo_alcalde[
        "porcentaje_claro"
    ]
    .min()
)

max_nulos_duracion_pct = (
    cps_alcaldia.groupby(
        "alcalde"
    )["duracion_dias"]
    .apply(
        lambda x: x.isna().mean() * 100
    )
    .max()
)

controles = [
    {
        "control": "ID contrato duplicado",
        "valor": n_ids_duplicados,
        "estado": (
            "PASS"
            if n_ids_duplicados == 0
            else "FAIL"
        )
    },
    {
        "control": "CPS estricto fuera de familia CPS",
        "valor": n_cps_fuera_familia,
        "estado": (
            "PASS"
            if n_cps_fuera_familia == 0
            else "FAIL"
        )
    },
    {
        "control": "ESE mezclada dentro de CPS de Alcaldia",
        "valor": n_ese_en_cps_alcaldia,
        "estado": (
            "PASS"
            if n_ese_en_cps_alcaldia == 0
            else "FAIL"
        )
    },
    {
        "control": "Inconsistencia Alcaldia entre NIT y flag",
        "valor": n_inconsistencia_alcaldia,
        "estado": (
            "PASS"
            if n_inconsistencia_alcaldia == 0
            else "FAIL"
        )
    },
    {
        "control": "ESAL clasificada con persona natural",
        "valor": n_esal_con_persona_natural,
        "estado": (
            "PASS"
            if n_esal_con_persona_natural == 0
            else "FAIL"
        )
    },
    {
        "control": "Proveedor sin clasificar",
        "valor": n_proveedor_revisar,
        "estado": (
            "PASS"
            if n_proveedor_revisar == 0
            else "WARNING"
        )
    },
    {
        "control": "Estado contractual por revisar",
        "valor": n_estados_revisar,
        "estado": (
            "PASS"
            if n_estados_revisar == 0
            else "WARNING"
        )
    },
    {
        "control": "Fechas invertidas",
        "valor": n_fechas_invertidas,
        "estado": (
            "PASS"
            if n_fechas_invertidas == 0
            else "WARNING"
        )
    },
    {
        "control": "Otros por revisar",
        "valor": n_otros_revisar,
        "estado": (
            "PASS"
            if n_otros_revisar <= 5
            else "WARNING"
        )
    },
    {
        "control": "Regimen especial persona natural por revisar",
        "valor": n_regimen_natural_revisar,
        "estado": (
            "PASS"
            if n_regimen_natural_revisar == 0
            else "WARNING"
        )
    },
    {
        "control": "Documentos sospechosos",
        "valor": n_documentos_sospechosos,
        "estado": (
            "PASS"
            if n_documentos_sospechosos == 0
            else "WARNING"
        )
    },
    {
        "control": "CPS probables fuera del estricto",
        "valor": n_cps_probables,
        "estado": (
            "PASS"
            if (
                n_cps_probables
                / max(len(cps_ampliado), 1)
                <= 0.01
            )
            else "WARNING"
        )
    },
    {
        "control": "Claridad minima de subtipo CPS por alcalde",
        "valor": min_claridad,
        "estado": (
            "PASS"
            if min_claridad >= 90
            else "WARNING"
        )
    },
    {
        "control": "Maximo porcentaje CPS sin duracion",
        "valor": round(
            max_nulos_duracion_pct,
            2
        ),
        "estado": (
            "PASS"
            if max_nulos_duracion_pct <= 2
            else "WARNING"
        )
    },
    {
        "control": "Contratos con duracion cero",
        "valor": n_duracion_cero,
        "estado": (
            "PASS"
            if n_duracion_cero == 0
            else "WARNING"
        )
    },
    {
        "control": "CPS con duracion mayor a 12 meses",
        "valor": n_cps_duracion_atipica,
        "estado": (
            "PASS"
            if n_cps_duracion_atipica == 0
            else "WARNING"
        )
    },
    {
        "control": "CPS con ejecucion parcial (valor mensual ajustado)",
        "valor": n_ejecucion_parcial,
        "estado": (
            "PASS"
            if n_ejecucion_parcial == 0
            else "WARNING"
        )
    },
    {
        "control": "CPS sin base confiable para valor mensual",
        "valor": n_sin_base_confiable,
        "estado": (
            "PASS"
            if n_sin_base_confiable == 0
            else "WARNING"
        )
    }
]

semaforo_calidad = pd.DataFrame(
    controles
)

if (
    semaforo_calidad["estado"]
    .eq("FAIL")
    .any()
):
    ESTADO_BASE = "NO APROBADA"
elif (
    semaforo_calidad["estado"]
    .eq("WARNING")
    .any()
):
    ESTADO_BASE = (
        "APROBADA CON ADVERTENCIAS CONTROLADAS"
    )
else:
    ESTADO_BASE = "APROBADA"

display(semaforo_calidad)

print(
    "Estado final de la base:",
    ESTADO_BASE
)

,control,valor,estado
0,ID contrato duplicado,0.00,PASS
1,CPS estricto fuera de familia CPS,0.00,PASS
2,ESE mezclada dentro de CPS de Alcaldia,0.00,PASS
3,Inconsistencia Alcaldia entre NIT y flag,0.00,PASS
4,ESAL clasificada con persona natural,0.00,PASS
5,Proveedor sin clasificar,0.00,PASS
6,Estado contractual por revisar,8.00,WARNING
7,Fechas invertidas,5.00,WARNING
8,Otros por revisar,3.00,PASS
9,Regimen especial persona natural por revisar,171.00,WARNING


Estado final de la base: APROBADA CON ADVERTENCIAS CONTROLADAS


In [71]:
# 48. Guardar auditorías

archivos_csv = {
    "02_auditoria_estados.csv": auditoria_estados,
    "02_documentos_varios_nombres.csv": documentos_con_varios_nombres,
    "02_documentos_varios_tipos.csv": documentos_con_varios_tipos,
    "02_nombres_varios_documentos.csv": nombres_con_varios_documentos,
    "02_revision_fechas_invertidas.csv": revision_fechas_invertidas,
    "02_revision_duracion_atipica.csv": revision_duracion_atipica,
    "02_revision_ejecucion_parcial.csv": revision_ejecucion_parcial,
    "02_resumen_fuente_valor_mensual.csv": resumen_ejecucion_parcial,
    "02_auditoria_valor_cero.csv": auditoria_valor_cero,
    "02_resumen_adiciones_cps.csv": resumen_adiciones_cps,
    "02_auditoria_esal.csv": auditoria_esal,
    "02_revision_servicios_por_objeto.csv": revision_servicios_por_objeto,
    "02_resumen_servicios_por_objeto_entidad.csv": resumen_servicios_por_objeto_entidad,
    "02_resumen_servicios_entidades.csv": resumen_servicios_entidades,
    "02_resumen_servicios_ese_anual.csv": resumen_servicios_ese_anual,
    "02_auditoria_regimen_especial_juridico.csv": auditoria_regimen_especial_juridico,
    "02_integridad_cps_familia.csv": integridad_cps_familia,
    "02_auditoria_tipo_familia.csv": auditoria_tipo_familia,
    "02_resumen_familias.csv": resumen_familias,
    "02_calidad_subtipo_alcalde.csv": calidad_subtipo_alcalde,
    "02_resumen_duracion_24m.csv": resumen_duracion_24m,
    "02_resumen_cps_subtipo_observado.csv": resumen_cps_subtipo_observado,
    "02_resumen_cps_subtipo_24m.csv": resumen_cps_subtipo_24m,
    "02_resumen_32m.csv": resumen_32m,
    "02_resumen_24m.csv": resumen_24m,
    "02_resumen_anio3_mismo_corte.csv": resumen_anio3,
    "02_resumen_recurrencia_24m.csv": resumen_recurrencia_24m,
    "02_distribucion_recurrencia_24m.csv": distribucion_recurrencia_24m,
    "02_limites_outliers.csv": limites_outliers,
    "02_semaforo_calidad.csv": semaforo_calidad
}

for nombre, tabla in archivos_csv.items():
    tabla.to_csv(
        RUTA_INTERMEDIOS / nombre,
        index=False,
        encoding="utf-8-sig"
    )

print("Auditorías guardadas")

Auditorías guardadas


In [72]:
# 49. Guardar bases finales

base_contratos.to_parquet(
    RUTA_INTERMEDIOS / "02_base_maestra_clasificada.parquet",
    index=False
)

base_analitica.to_parquet(
    RUTA_PROCESADOS / "02_base_analitica_valida.parquet",
    index=False
)

cps.to_parquet(
    RUTA_PROCESADOS / "02_cps_personas_naturales.parquet",
    index=False
)

cps_ampliado.to_parquet(
    RUTA_PROCESADOS / "02_cps_ampliado_sensibilidad.parquet",
    index=False
)

contratistas_empresariales.to_parquet(
    RUTA_PROCESADOS / "02_empresas_y_grupos_contratistas.parquet",
    index=False
)


servicios_persona_natural_entidades.to_parquet(
    RUTA_PROCESADOS / "02_servicios_persona_natural_entidades.parquet",
    index=False
)

servicios_ese.to_parquet(
    RUTA_PROCESADOS / "02_servicios_persona_natural_ese_barrancabermeja.parquet",
    index=False
)

print("Bases finales guardadas")

Bases finales guardadas


In [73]:
# 50. Guardar manifiesto metodológico

manifiesto = {
    "version_base": "02_v5",
    "corte_datos": "2026-09-06",
    "unidad_analisis": "Contrato digital SECOP II",
    "entidades": "La Alcaldia se identifica por NIT 890201900. La Empresa Social del Estado Barrancabermeja se identifica por NIT 829001846 y se analiza como entidad separada, igual que las demas entidades locales.",
    "servicios_persona_natural_entidades": "Se crea un universo adicional que combina contratos tipificados por SECOP como Prestacion de servicios y contratos cuyo objeto dice explicitamente PRESTACION DE SERVICIOS aunque SECOP use otra categoria. Este universo se analiza por entidad y no se mezcla automaticamente con los CPS de la Alcaldia.",
    "identidad_proveedor": "Documento del proveedor como llave principal. Nombre canonico solo para presentacion.",
    "universo_cps_principal": "Prestacion de servicios, persona natural, evidencia de profesional o apoyo a la gestion y contrato valido.",
    "familia_contractual": "Se prioriza CPS antes de clasificaciones textuales. Tipo de contrato SECOP es la fuente principal para obra, bienes, seguros, consultoria y otras familias.",
    "politica_nulos": "No imputar fechas, valores, duracion ni subtipo CPS. Excluir solo de la metrica afectada.",
    "politica_estados": "Excluir borrador, cancelado, anulado, eliminado, rechazado, desistido, revocado y abortado. Estados no resueltos no entran a la base analitica.",
    "subtipo_cps": "Profesional y apoyo solo se comparan cuando la evidencia textual es clara. Casos ambiguos se conservan sin imputacion.",
    "esal": "ESAL significa Entidad sin animo de lucro. La familia ESAL solo admite persona juridica o grupo. Las personas naturales no se clasifican como ESAL.",
    "regimen_especial": "Contratos de regimen especial con persona juridica o grupo se mantienen fuera del universo CPS.",
    "duracion": "Duracion registrada entre fecha de inicio y fecha de fin. Dias adicionados se auditan por separado y no se suman automaticamente.",
    "valor_mensual": "Se usa valor_mensual_ajustado. SECOP registra la fecha de fin REAL, de modo que un contrato terminado anticipadamente queda con duracion corta pero conserva el valor total pactado; dividir uno entre otro infla el valor mensual hasta tres veces. Si el contrato tiene estado de cierre definitivo (cerrado, terminado o liquidado) y quedo mas del 5 por ciento sin ejecutar, el valor mensual se calcula sobre el valor ejecutado, que corresponde al mismo periodo que la duracion registrada. La condicion de cierre definitivo es indispensable: el reporte de pagos llega con rezago, y entre contratos no cerrados la ejecucion parcial cae de 50 a 15 por ciento segun la antiguedad, mientras que entre los cerrados es estable en torno al 7 por ciento e igual para ambos alcaldes. Sin esa condicion se castigaria a la administracion con contratos mas recientes. Los contratos cerrados sin valor ejecutado reportado se marcan como no confiables y quedan fuera del analisis de valores.",
    "valores": "Valores nominales. Comparaciones economicas entre gobiernos requieren IPC.",
    "duracion_no_analizable": "Se marcan (no se borran) los contratos con duracion cero -fecha de fin no diligenciada- y los CPS de mas de 12 meses -probable error en el ano de la fecha de fin-. Quedan excluidos de las estadisticas de duracion y valor mensual.",
    "outliers": "Los valores extremos se marcan por entidad, subtipo y ano. Nunca se eliminan automaticamente.",
    "anio_gobierno": {
        "inicio_mandatos": {n: str(f.date()) for n, f in INICIO_MANDATO.items()},
        "variables_creadas": ["mes_gobierno", "anio_gobierno"],
        "nota": "Posicion del contrato dentro del mandato (mes 1 a 48). Permite comparar fases equivalentes de gobierno en lugar de anos calendario."
    },
    "calendario_electoral": {
        "elecciones_territoriales": [str(e.date()) for e in ELECCIONES_TERRITORIALES],
        "meses_restriccion_garantias": MESES_RESTRICCION_GARANTIAS,
        "variables_creadas": ["periodo_electoral", "periodo_preelectoral", "tipo_anio_electoral", "ventana_ley_garantias", "dias_a_prox_eleccion"],
        "nota": "Ventana de ley de garantias = 4 meses previos a cada eleccion territorial. No constituye un juicio de cumplimiento normativo sobre ningun contrato individual."
    },
    "ventanas_comparables": {
        "32_meses": "Alfonso abr-2021 a nov-2023; Jonathan ene-2024 a ago-2026",
        "24_meses": "Alfonso ene-2022 a dic-2023; Jonathan ene-2024 a dic-2025",
        "tercer_anio_mismo_corte": "Alfonso ene-2022 a 6-sep-2022; Jonathan ene-2026 a 6-sep-2026"
    },
    "estado_base": ESTADO_BASE
}

with open(
    RUTA_INTERMEDIOS / "02_manifiesto_metodologico.json",
    "w",
    encoding="utf-8"
) as archivo:
    json.dump(
        manifiesto,
        archivo,
        ensure_ascii=False,
        indent=2
    )

print("Manifiesto guardado")

Manifiesto guardado


In [74]:
# 51. Control final

print("=" * 70)
print(f"Registros originales: {len(df):,}")
print(f"Contratos únicos: {len(base_contratos):,}")
print(f"Base analítica válida: {len(base_analitica):,}")
print(f"CPS estricto: {len(cps):,}")
print(f"Personas únicas CPS: {cps['proveedor_llave'].nunique():,}")
print(f"Empresas y grupos: {contratistas_empresariales['proveedor_llave'].nunique():,}")
print(f"Servicios persona natural por entidad: {len(servicios_persona_natural_entidades):,}")

Registros originales: 37,574
Contratos únicos: 37,574
Base analítica válida: 37,566
CPS estricto: 32,778
Personas únicas CPS: 9,926
Empresas y grupos: 712
Servicios persona natural por entidad: 34,692
